# 04 — Modelos: Small CNN na imagem e baselines tabulares

Notebook equivalente a `src/models/`. Treina quatro modelos **nas mesmas 5 partições** (`coorte_cv.csv`) e
com **a mesma padronização** (estatística só do treino de cada fold):

| modelo | entrada | o que testa |
|---|---|---|
| `cnn_hilbert` | imagem `(133, 134, 3)` na curva de Hilbert | o formato principal |
| `cnn_vetor` | vetor `(17.818, 3)` na ordem do genoma | a mesma rede em 1D: a dobra 2D ajuda? |
| `logistica` | os 53.454 números do tensor, sem formato | linear, sem imagem |
| `xgboost` | os mesmos 53.454 números | árvores, sem imagem |

Decisões do usuário (2026-10-09): mutação em `log(1 + contagem)` antes de padronizar (feita no `03_images`
e registrada no `padronizacao.json`); rede em **Keras** (motor TensorFlow-CPU); a rede é a **Small CNN** do
projeto irmão; baselines logística e XGBoost já neste notebook.

Entrada: `data/processed/`, gerado pelo `03_images`. Saída: `results/predicoes_oof.csv`, com a probabilidade
de cada classe para cada paciente, prevista pelo modelo do fold em que ele estava no teste. Progresso da execução em
`results/04_models.log` (fold, época e memória). A avaliação
completa (IC por bootstrap, regra do TCGA, ablações por canal) fica para o `05_eval`.

In [1]:
import gc
import json
import os
import time
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # silencia o log C++ do TF (avisos de CPU e de versão do tf.data); erro Python continua aparecendo
import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             matthews_corrcoef)
from sklearn.model_selection import StratifiedShuffleSplit
from xgboost import XGBClassifier

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
PROCESSED, RESULTS = RAIZ / "data" / "processed", RAIZ / "results"
SEED, K = 42, 5
CLASSES = ["CIN", "MSI", "GS", "EBV"]  # = classe_idx 0, 1, 2, 3
CANAIS = ["mutacao", "cnv", "metilacao"]
PROBS = [f"p_{c}" for c in CLASSES]
tf.config.experimental.enable_op_determinism()  # mesma semente, mesmo resultado

# o nbconvert só grava o notebook no fim: o progresso vai para um arquivo (`tail -f results/04_models.log`),
# com a memória do processo, porque a máquina tem 4,8 GB e a 1ª execução morreu por falta de memória
LOG = RESULTS / "04_models.log"


def log(msg, tela=True):
    mem = {l.split(":")[0]: int(l.split()[1]) // 1024 for l in open("/proc/self/status") if l.startswith(("VmRSS", "VmHWM"))}
    linha = f"{time.strftime('%H:%M:%S')} [RAM {mem['VmRSS']} MB, pico {mem['VmHWM']} MB] {msg}"
    with open(LOG, "a") as f:
        f.write(linha + "\n")
    if tela:
        print(linha)


ACOMP = RESULTS / "04_models_acompanhamento.log"  # 2º log, só para acompanhar o treino (mural #65): nenhuma decisão usa este arquivo


def acomp(msg):
    with open(ACOMP, "a", encoding="utf-8") as f:
        f.write(msg + "\n")


def virgula(x, sinal=False):
    return (f"{x:+.2f}" if sinal else f"{x:.2f}").replace(".", ",")


def matriz(y_, yp):
    return confusion_matrix(y_, yp, labels=range(len(CLASSES)))  # linha = real, coluna = previsto; CIN = 0, GS = 2


def f1_cm(cm):
    return 2 * np.diag(cm) / np.maximum(cm.sum(axis=0) + cm.sum(axis=1), 1)


def texto_f1(cm):
    # ("pior 0,72 (GS)", "CIN 0,93 MSI 1,00 GS 0,72 EBV 0,91")
    f1 = f1_cm(cm)
    return f"pior {virgula(f1.min())} ({CLASSES[f1.argmin()]})", " ".join(f"{c} {virgula(v)}" for c, v in zip(CLASSES, f1))


assert texto_f1(np.array([[8, 0, 2, 0], [0, 3, 0, 0], [1, 0, 1, 0], [0, 0, 0, 1]]))[0] == "pior 0,40 (GS)"  # F1 do GS: 2·1 / (2 + 3)
log("=== nova execução ===", tela=False)
acomp(f"=== nova execução: {time.strftime('%F %T')} ===")


coorte = pd.read_csv(PROCESSED / "coorte_cv.csv")
tensor = np.load(PROCESSED / "tensor_bruto.npy")
padronizacao = {int(k): v for k, v in json.loads((PROCESSED / "padronizacao.json").read_text()).items()}
layout = pd.read_csv(PROCESSED / "layout_hilbert.csv")
y, fold = coorte.classe_idx.to_numpy(), coorte.fold.to_numpy()
assert tensor.shape == (len(coorte), 3, len(layout)) and set(padronizacao) == set(range(K))
assert all(s["mutacao"].get("transformacao") == "log1p" for s in padronizacao.values())  # json do 03_images atual
print(f"{len(coorte)} pacientes {dict(zip(CLASSES, np.bincount(y)))} · tensor {tensor.shape} · keras {keras.__version__}")

336 pacientes {'CIN': np.int64(202), 'MSI': np.int64(59), 'GS': np.int64(46), 'EBV': np.int64(29)} · tensor (336, 3, 17818) · keras 3.15.1


## Padronização e imagem — as mesmas funções do `03_images`

Copiadas, porque o notebook é autocontido. A conferência garante que a cópia bate com o `padronizacao.json`:
no treino de cada fold, todo canal fica com média ≈ 0. Se o `log1p` faltasse, a mutação sairia deslocada.

In [2]:
def transformar(x):
    x = x.copy()
    x[:, 0] = np.log1p(x[:, 0])  # mutação em log(1 + contagem), como no 03_images
    return x


def padronizar(x, stats):
    x = transformar(x)
    media = np.array([stats[c]["media"] for c in CANAIS], np.float32)[None, :, None]
    desvio = np.array([stats[c]["desvio"] for c in CANAIS], np.float32)[None, :, None]
    return np.nan_to_num((x - media) / desvio, nan=0.0)


FORMA = (layout.linha.max() + 1, layout.coluna.max() + 1)


def para_hilbert(x):
    img = np.zeros((x.shape[0], x.shape[1], *FORMA), np.float32)
    img[:, :, layout.linha, layout.coluna] = x
    return img


for k in range(K):
    assert np.allclose(padronizar(tensor[fold != k], padronizacao[k]).mean(axis=(0, 2)), 0, atol=1e-3), k
assert FORMA == (133, 134) and not layout.duplicated(["linha", "coluna"]).any()

## Small CNN em Keras

A Small CNN do projeto irmão (`proj_mutacoes_classificacao_TCGA-STAD/src/models/cnn_classifier.py`, em torch),
reescrita em Keras. São 4 blocos *convolução 3 → BatchNorm → ReLU → MaxPool 2*, com 16 → 32 → 64 → 64 filtros,
seguidos de média global, Dropout 0,3 e uma camada densa de 4 saídas. A BatchNorm usa `momentum=0.9`, o
equivalente ao padrão do torch. A mesma função monta a versão 2D
(imagem) e a 1D (vetor). Na 1D, o filtro tem 3 posições em vez de 3×3, por isso ela tem menos parâmetros.

O Keras usa os canais **por último**: a imagem `(3, 133, 134)` entra como `(133, 134, 3)`.

Treino com os hiperparâmetros do irmão:
- **Validação interna:** 15% do treino, estratificada, usada só para decidir quando parar. O fold de
  teste nunca é visto;
- **Parada antecipada:** pelo F1 macro da validação interna, paciência de 15 épocas, até 60 épocas. Os
  pesos voltam para os da melhor época;
- **Otimização:** Adam com lr 1e-3, batch de 32, perda de entropia cruzada com peso inverso à frequência
  da classe;
- **Sem aumento de dados:** girar ou espelhar a imagem tiraria cada gene do seu pixel.

In [3]:
def small_cnn(entrada, n_classes=4, dropout=0.3, filtros=(16, 32, 64, 64), achatar=False):
    conv, pool, gap = ((layers.Conv2D, layers.MaxPooling2D, layers.GlobalAveragePooling2D) if len(entrada) == 3
                       else (layers.Conv1D, layers.MaxPooling1D, layers.GlobalAveragePooling1D))
    x = entrada_ = keras.Input(entrada)
    for f in filtros:
        x = conv(f, 3, padding="same")(x)
        x = layers.BatchNormalization(momentum=0.9)(x)  # = momentum 0,1 do torch; o 0,99 padrão do Keras demora a assentar
        x = layers.ReLU()(x)
        x = pool(2)(x)
    x = layers.Dropout(dropout)((layers.Flatten() if achatar else gap())(x))  # achatar: R9 da PARTE C (saída por posição)
    return keras.Model(entrada_, layers.Dense(n_classes)(x))  # logits; a perda aplica o softmax


class F1MacroVal(keras.callbacks.Callback):
    # põe o F1 macro da validação interna nos logs; vem antes do EarlyStopping na lista, que lê o mesmo dicionário
    def __init__(self, x, y):
        super().__init__()
        self.x, self.y, self.f1s = x, y, []

    def on_epoch_end(self, epoch, logs=None):
        yp = self.model.predict(self.x, verbose=0).argmax(axis=1)
        self.f1s.append(f1_score(self.y, yp, average="macro"))
        logs["val_f1_macro"] = self.f1s[-1]
        log(f"    época {epoch + 1}: perda {logs['loss']:.3f} · F1 macro val {self.f1s[-1]:.3f}", tela=False)
        if self.params["epochs"] > 1:  # os autotestes de 1 época ficam fora do log de acompanhamento
            cm = matriz(self.y, yp)
            pior, por_classe = texto_f1(cm)
            acomp(f"  época {epoch + 1} · validação: macro {virgula(self.f1s[-1])} · {por_classe} · {pior} · CIN→GS {cm[0, 2]} · GS→CIN {cm[2, 0]}")


def pesos_por_classe(y_):
    contagem = np.bincount(y_, minlength=len(CLASSES))
    return len(y_) / (len(CLASSES) * contagem)


def treinar_cnn(x_tr, y_tr, x_te, seed, guardar=None):
    keras.backend.clear_session()  # sem isso, cada modelo novo acumula memória no TensorFlow
    gc.collect()
    tr, va = next(StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed).split(x_tr, y_tr))
    keras.utils.set_random_seed(seed)
    modelo = small_cnn(x_tr.shape[1:])
    modelo.compile(keras.optimizers.Adam(1e-3), keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    f1_val = F1MacroVal(x_tr[va], y_tr[va])
    parada = keras.callbacks.EarlyStopping("val_f1_macro", mode="max", patience=15, restore_best_weights=True)
    modelo.fit(x_tr[tr], y_tr[tr], epochs=60, batch_size=32, class_weight=dict(enumerate(pesos_por_classe(y_tr[tr]))),
               callbacks=[f1_val, parada], verbose=0)
    log(f"  {len(f1_val.f1s)} épocas, melhor F1 macro na validação interna {max(f1_val.f1s):.3f}")
    if guardar:
        modelo.save(guardar)  # bloco C do 05_eval: o Grad-CAM precisa do modelo treinado
    return softmax(modelo.predict(x_te, verbose=0), axis=1)


print({nome: small_cnn(e).count_params() for nome, e in {"2D (Hilbert)": (*FORMA, 3), "1D (vetor)": (len(layout), 3)}.items()})

{'2D (Hilbert)': 61476, '1D (vetor)': 21252}


## Baselines tabulares

Mesma entrada da Small CNN, achatada: os 3 canais × 17.818 genes viram 53.454 colunas, sem formato de
imagem. Configurações iguais às do irmão (`src/eval/baselines.py`):
- **logística:** L2, C = 1, peso balanceado por classe;
- **XGBoost:** 400 árvores, profundidade 4, lr 0,1, subamostra 0,8, metade das colunas por árvore e peso
  inverso à frequência da classe.

In [4]:
def treinar_logistica(x_tr, y_tr, x_te, seed):
    clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, random_state=seed)
    return clf.fit(x_tr, y_tr).predict_proba(x_te)


def treinar_xgboost(x_tr, y_tr, x_te, seed):
    clf = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.5,
                        tree_method="hist", random_state=seed, n_jobs=-1)
    return clf.fit(x_tr, y_tr, sample_weight=pesos_por_classe(y_tr)[y_tr]).predict_proba(x_te)

## Validação cruzada

Para cada fold `k`: padroniza **o tensor inteiro com a estatística do treino de `k`**, treina nos outros 4
folds e prevê o fold `k`. Ao fim, cada paciente tem uma previsão por modelo, feita por um modelo que nunca o
viu.

**Retomada:** cada fold de cada modelo é gravado em `results/oof/` assim que termina. Se a execução cair (a
1ª morreu por falta de memória, com 4,8 GB na máquina), basta rodar de novo: o que já existe é reaproveitado,
e o processo novo começa com a memória zerada. Para treinar tudo de novo, apague `results/oof/`.

In [5]:
MODELOS = {
    "cnn_hilbert": (lambda z: np.moveaxis(para_hilbert(z), 1, -1), treinar_cnn),  # (n, 133, 134, 3)
    "cnn_vetor": (lambda z: np.moveaxis(z, 1, -1), treinar_cnn),                  # (n, 17818, 3)
    "logistica": (lambda z: z.reshape(len(z), -1), treinar_logistica),            # (n, 53454)
    "xgboost": (lambda z: z.reshape(len(z), -1), treinar_xgboost),
}

OOF = RESULTS / "oof"  # uma previsão por modelo e fold, gravada ao terminar; apague a pasta para treinar tudo de novo
OOF.mkdir(exist_ok=True)
pred = []
for nome, (formatar, treinar) in MODELOS.items():
    t0 = time.time()
    for k in range(K):
        arq = OOF / f"{nome}_fold{k}.csv"
        if arq.exists():
            log(f"{nome} · fold {k}: reaproveitado de {arq.name}")
        else:
            log(f"{nome} · fold {k}")
            x = formatar(padronizar(tensor, padronizacao[k]))
            tr, te = fold != k, fold == k
            proba = treinar(x[tr], y[tr], x[te], SEED + k)
            pd.DataFrame(proba, columns=PROBS).assign(
                modelo=nome, amostra=coorte.amostra[te].to_numpy(), fold=k, classe_idx=y[te]).to_csv(arq, index=False)
        pred.append(pd.read_csv(arq))
    log(f"{nome} terminado em {(time.time() - t0) / 60:.1f} min")

pred = pd.concat(pred, ignore_index=True)
assert (pred.groupby("modelo").amostra.nunique() == len(coorte)).all() and len(pred) == len(MODELOS) * len(coorte)
assert np.allclose(pred[PROBS].sum(axis=1), 1, atol=1e-4)
pred.to_csv(RESULTS / "predicoes_oof.csv", index=False)
log("fim: results/predicoes_oof.csv gravado")

22:41:07 [RAM 759 MB, pico 984 MB] cnn_hilbert · fold 0: reaproveitado de cnn_hilbert_fold0.csv
22:41:07 [RAM 759 MB, pico 984 MB] cnn_hilbert · fold 1: reaproveitado de cnn_hilbert_fold1.csv


22:41:07 [RAM 759 MB, pico 984 MB] cnn_hilbert · fold 2: reaproveitado de cnn_hilbert_fold2.csv
22:41:07 [RAM 760 MB, pico 984 MB] cnn_hilbert · fold 3: reaproveitado de cnn_hilbert_fold3.csv


22:41:07 [RAM 760 MB, pico 984 MB] cnn_hilbert · fold 4: reaproveitado de cnn_hilbert_fold4.csv
22:41:07 [RAM 760 MB, pico 984 MB] cnn_hilbert terminado em 0.0 min
22:41:07 [RAM 760 MB, pico 984 MB] cnn_vetor · fold 0: reaproveitado de cnn_vetor_fold0.csv


22:41:07 [RAM 760 MB, pico 984 MB] cnn_vetor · fold 1: reaproveitado de cnn_vetor_fold1.csv
22:41:07 [RAM 760 MB, pico 984 MB] cnn_vetor · fold 2: reaproveitado de cnn_vetor_fold2.csv


22:41:07 [RAM 760 MB, pico 984 MB] cnn_vetor · fold 3: reaproveitado de cnn_vetor_fold3.csv
22:41:08 [RAM 760 MB, pico 984 MB] cnn_vetor · fold 4: reaproveitado de cnn_vetor_fold4.csv
22:41:08 [RAM 760 MB, pico 984 MB] cnn_vetor terminado em 0.0 min


22:41:08 [RAM 760 MB, pico 984 MB] logistica · fold 0: reaproveitado de logistica_fold0.csv
22:41:08 [RAM 760 MB, pico 984 MB] logistica · fold 1: reaproveitado de logistica_fold1.csv


22:41:08 [RAM 760 MB, pico 984 MB] logistica · fold 2: reaproveitado de logistica_fold2.csv
22:41:08 [RAM 760 MB, pico 984 MB] logistica · fold 3: reaproveitado de logistica_fold3.csv
22:41:08 [RAM 760 MB, pico 984 MB] logistica · fold 4: reaproveitado de logistica_fold4.csv


22:41:08 [RAM 760 MB, pico 984 MB] logistica terminado em 0.0 min
22:41:08 [RAM 760 MB, pico 984 MB] xgboost · fold 0: reaproveitado de xgboost_fold0.csv


22:41:08 [RAM 760 MB, pico 984 MB] xgboost · fold 1: reaproveitado de xgboost_fold1.csv
22:41:08 [RAM 760 MB, pico 984 MB] xgboost · fold 2: reaproveitado de xgboost_fold2.csv
22:41:08 [RAM 760 MB, pico 984 MB] xgboost · fold 3: reaproveitado de xgboost_fold3.csv


22:41:08 [RAM 760 MB, pico 984 MB] xgboost · fold 4: reaproveitado de xgboost_fold4.csv
22:41:08 [RAM 760 MB, pico 984 MB] xgboost terminado em 0.0 min


22:41:09 [RAM 762 MB, pico 984 MB] fim: results/predicoes_oof.csv gravado


## Resultado — painel por classe

Métricas sobre as previsões dos 5 folds juntas (336 pacientes; n por classe: CIN 202 · MSI 59 · GS 46 ·
EBV 29). **Pior F1** é o F1 da classe que foi pior: é o número da equidade, o que a pergunta de pesquisa
cobra. O AUC-PR de um modelo que chuta fica na prevalência da classe: CIN 0,60 · MSI 0,18 · GS 0,14 ·
EBV 0,09.

Barra de referência do irmão (só mutação, mesma coorte): regra de marcadores MCC 0,660 · XGBoost 0,655 ·
melhor CNN 0,561; GS nunca passou de F1 ≈ 0,5.

In [6]:
def metricas(d):
    p, yt = d[PROBS].to_numpy(), d.classe_idx.to_numpy()
    yp = p.argmax(axis=1)
    f1 = f1_score(yt, yp, average=None, labels=range(len(CLASSES)))
    return {"MCC": matthews_corrcoef(yt, yp), "acurácia balanceada": balanced_accuracy_score(yt, yp),
            "F1 macro": f1.mean(), "pior F1": f1.min(), **{f"F1 {c}": v for c, v in zip(CLASSES, f1)},
            **{f"AUC-PR {c}": average_precision_score(yt == i, p[:, i]) for i, c in enumerate(CLASSES)}}


pd.DataFrame({m: metricas(d) for m, d in pred.groupby("modelo", sort=False)}).T.round(3)

,MCC,acurácia balanceada,F1 macro,pior F1,F1 CIN,F1 MSI,F1 GS,F1 EBV,AUC-PR CIN,AUC-PR MSI,AUC-PR GS,AUC-PR EBV
cnn_hilbert,0.824,0.913,0.874,0.694,0.907,1.000,0.694,0.897,0.986,1.000,0.830,0.944
cnn_vetor,0.819,0.919,0.874,0.695,0.901,0.983,0.695,0.918,0.979,0.999,0.766,0.988
logistica,0.842,0.916,0.895,0.750,0.924,0.956,0.750,0.951,0.974,0.988,0.788,0.992
xgboost,0.810,0.873,0.872,0.674,0.916,0.930,0.674,0.967,0.972,0.962,0.774,1.000


In [7]:
def confusao(d):
    return pd.DataFrame(confusion_matrix(d.classe_idx, d[PROBS].to_numpy().argmax(axis=1), labels=range(len(CLASSES))),
                        index=pd.Index(CLASSES, name="real"), columns=CLASSES)


pd.concat({m: confusao(d) for m, d in pred.groupby("modelo", sort=False)}, axis=1)  # linhas = real, colunas = previsto

cnn_hilbert             cnn_vetor             logistica              \
             CIN MSI  GS EBV       CIN MSI  GS EBV       CIN MSI  GS EBV   
real                                                                       
CIN          170   0  30   2       169   0  30   3       182   0  18   2   
MSI            0  59   0   0         0  58   1   0         4  54   1   0   
GS             3   0  42   1         4   0  41   1         6   0  39   1   
EBV            0   0   3  26         0   1   0  28         0   0   0  29   

     xgboost              
         CIN MSI  GS EBV  
real                      
CIN      186   2  13   1  
MSI        4  53   2   0  
GS        14   0  31   1  
EBV        0   0   0  29

## Leitura (execução de 2026-10-09; um fold de teste por paciente, uma semente, ainda sem IC)

- **Os quatro modelos ficam na mesma faixa:** MCC 0,81–0,84. O irmão, só com mutação, ficou em 0,66 (regra de
  marcadores) e 0,56 (melhor CNN). CNV e metilação acrescentam muito, como a circularidade previa.
- **A pior classe é GS em todos os modelos:** F1 0,67–0,75. O erro dominante é **CIN previsto como GS**: 30 dos
  202 CIN nas duas Small CNN, 18 na logística, 13 no XGBoost. São CIN de aneuploidia baixa, a fronteira que o TCGA
  traçou pelo grau de aneuploidia.
- **MSI saiu perfeito na Small CNN (Hilbert)** (59/59, AUC-PR 1,000). EBV fica entre 0,90 e 0,97.
- **Hilbert × vetor 1D:** praticamente empatados no F1. A Hilbert tem AUC-PR de GS maior (0,830 × 0,766), ou seja,
  ordena melhor os GS, mas o ponto de corte (argmax) é o mesmo nas duas.
- **A logística, o modelo mais simples, tem o melhor pior-F1 (0,750) e o melhor MCC.** As assinaturas do rótulo
  estão em poucos genes de efeito forte, e um modelo linear as captura.
- Diferenças de 0,02 a 0,05 entre modelos, com GS = 46 e EBV = 29, provavelmente estão dentro do ruído. Quem
  decide é o `05_eval`, com bootstrap pareado, a regra do TCGA e as ablações por canal.
- **Custo:** Small CNN (Hilbert) 10 min, Small CNN (vetor 1D) 25 min, logística 0,6 min, XGBoost 22 min. O XGBoost
  chegou a 3,9 GB de pico, no limite da máquina.

## Conferência: MSI = 1,0 na Small CNN (Hilbert) é sobreajuste?

Pergunta do usuário (2026-10-09). **Sobreajuste** é o modelo decorar o treino e errar em quem não viu. Aqui, toda
métrica vem de pacientes que o modelo **não viu** (fora do fold), então o sobreajuste puxaria o número para
baixo, não para cima. Um 1,0 fora do fold levanta duas outras suspeitas:
1. **Vazamento:** o paciente de teste influenciou o treino de algum jeito;
2. **Tarefa fácil por construção:** a classe se separa por uma variável crua, sem precisar aprender nada.

A célula abaixo confere as duas. Usa só o dado **bruto, sem treino nenhum**: a carga mutacional e o beta do
promotor de MLH1, as duas marcas do MSI no TCGA 2014.

In [8]:
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score

# 1. vazamento: um paciente = uma amostra = um fold de teste (os demais pontos estão conferidos acima e no 03_images)
assert coorte.paciente.is_unique and (pred.groupby("modelo").amostra.nunique() == len(coorte)).all()

# 2. separação por variável crua, sem nenhum modelo
msi = y == 1
crus = {"genes mutados": (tensor[:, 0] > 0).sum(axis=1),               # genes com variante não silenciosa
        "beta MLH1": tensor[:, 2, layout.gene.tolist().index("MLH1")]}  # promotor de MLH1
h = pred[pred.modelo == "cnn_hilbert"]
ic = binomtest(int(msi.sum()), int(msi.sum())).proportion_ci()
print(f"Small CNN (Hilbert), p_MSI: menor entre os MSI {h[h.classe_idx == 1].p_MSI.min():.3f} · "
      f"maior entre os outros {h[h.classe_idx != 1].p_MSI.max():.3f}")
print(f"recall de MSI {msi.sum()}/{msi.sum()}: IC 95% (Clopper-Pearson) {ic.low:.3f}–{ic.high:.3f}")
pd.DataFrame({nome: {"AUC MSI × resto (sem treino)": roc_auc_score(msi, v), "MSI: mínimo": v[msi].min(),
                     "MSI: mediana": np.median(v[msi]), "outros: mediana": np.median(v[~msi]),
                     "outros: máximo": v[~msi].max()} for nome, v in crus.items()}).round(3)

Small CNN (Hilbert), p_MSI: menor entre os MSI 0.492 · maior entre os outros 0.277
recall de MSI 59/59: IC 95% (Clopper-Pearson) 0.939–1.000


,genes mutados,beta MLH1
AUC MSI × resto (sem treino),0.998,0.931
MSI: mínimo,364.000,0.084
MSI: mediana,980.000,0.525
outros: mediana,83.000,0.098
outros: máximo,857.000,0.587


**Leitura:**
- **Não há sinal de vazamento.** Cada paciente está num fold de teste só. A padronização usa só o treino, e a
  parada antecipada usa uma validação tirada do treino. Os hiperparâmetros vieram do irmão, sem ajuste
  olhando o teste.
- **O MSI é quase separável por um número cru.** A contagem de genes mutados, sem modelo nenhum, separa MSI do
  resto com AUC 0,998: mediana de 980 genes no MSI contra 83 nos outros. É a hipermutação, consequência direta
  da instabilidade de microssatélites, que foi o critério do rótulo. Mesmo assim, alguns não MSI chegam a 857
  genes e o MSI mais baixo tem 364: a carga sozinha erra alguns. O MLH1 sozinho fica em AUC 0,931.
- A Small CNN (Hilbert) junta as duas pistas e separa com folga: o MSI menos confiante tem p = 0,49, e o não MSI mais
  confiante, p = 0,28.
- **O 1,0 é esperado pela circularidade, não é sobreajuste.** A barra clínica no MSI também é essencialmente
  perfeita (SKUBLENY 2025, `referencias_literatura.md` §2).
- **Mas 59 de 59 não prova 100%.** O IC 95% do recall vai de 0,939 a 1,000: com outra amostra de pacientes,
  a Small CNN poderia errar 1 ou 2 MSI em 59. As classes difíceis continuam sendo CIN × GS.

# Otimização da Small CNN (2026-10-31)

Decisão do usuário (2026-10-30 e 2026-10-31): otimizar a Small CNN com os itens 1, 2 e 4 da recomendação de
`docs/relatorio_otimizacao.txt` (seção 7), nesta ordem, e ver qual variante fica melhor. Ficam de fora o item 3 (CoordConv,
GAP + GMP, sem o último MaxPool) e o z por gene: no `05_eval` (bloco D), embaralhar os genes nos pixels não mudou nada, e o z
por gene piorou o GS.

O diagnóstico do relatório (seção 1.2): depois da média global, a Small CNN sabe **quanto** de cada padrão há na imagem, mas não
**onde**. Por isso ela não usa o sinal de um gene só (MLH1, CDH1, RHOA), que a logística usa com um peso por gene e canal.

| item | o que é | pergunta |
|---|---|---|
| 1 | pré-registro e BBC-CV | quanto do "ganho" da melhor variante é só a sorte de ter escolhido a melhor? |
| 2 | ramo linear ao lado da Small CNN (*wide & deep*) | dar à rede um peso por pixel recupera o sinal de gene isolado? |
| 4 | AdamW forte, focal loss e group DRO | treinar para o pior caso melhora o GS? |

**Pré-registro** (gravado no diário em 2026-10-31, antes de treinar):
- **métrica principal:** pior F1 no argmax, nas 336 previsões fora do fold, como média das 5 sementes (42, 142, 242, 342, 442;
  no fold `k`, a rede usa `semente + k`). Secundárias: F1 por classe, MCC, AUC-PR do GS e o pior F1 da média das
  probabilidades das 5 sementes (o *ensemble*);
- tudo igual à Small CNN (Hilbert) de cima, salvo a mudança de cada variante;
- **item 2:** `largo` (só o ramo linear: o controle sem convolução), `profundo` (a Small CNN atual, com as sementes já
  gravadas) e `largo_profundo` (os dois somados nos logits). O L2 do ramo linear é 1/(2n), o equivalente ao C = 1 da logística;
- **item 4**, sobre a melhor entre `profundo` e `largo_profundo` (maior média do pior F1; em empate, `profundo`): `adamw`
  (decaimento de peso 1,0), `adamw_focal` (focal loss, γ = 2) e `adamw_dro` (group DRO por classe, η = 0,01);
- **todas as variantes são reportadas**, e a melhor é corrigida pela BBC-CV.

**Retomada:** cada fold de cada variante e semente vai para `results/oof_otim/` assim que termina. O `profundo` e a Small CNN
(vetor 1D) reaproveitam as sementes já gravadas (`results/oof/` para a semente 42; `results/oof_b/`, do bloco B do `05_eval`,
para as outras). Se faltarem, são treinadas aqui. Para treinar tudo de novo, apague `results/oof_otim/`.

Custo previsto (8 CPUs, sem GPU): 25 rodadas de 5 folds. São 5 do `largo`, de poucos minutos cada, e 20 do `largo_profundo` e do
item 4, de 10 a 15 min cada, como a Small CNN. Total ≈ 4 a 5 h. Progresso em `results/04_models.log`.

## Item 1 — avaliação honesta: pré-registro e BBC-CV

**O problema.** Treinar várias variantes e ficar com a melhor nos mesmos 336 pacientes infla o resultado. Já aconteceu aqui: com
o limiar de GS, a melhor das 5 sementes deu 0,771 de pior F1, e a média das 5 ficou em 0,669. A conferência abaixo mostra o
tamanho do efeito: entre 20 configurações **de mesma qualidade** (85% de acerto ao acaso), a maior fica bem acima da média
sem ser melhor em nada.

**A primeira proteção é o pré-registro:** as variantes, a métrica e a regra de escolha foram escritas no diário antes do treino,
e todas são reportadas.

**A segunda é a BBC-CV** (*bootstrap bias corrected CV*; TSAMARDINOS et al., 2018; relatório, seção 2.3). Ela usa só as
previsões fora do fold que já existem, sem treinar nada a mais. Em cada um dos 2.000 sorteios do bootstrap:
1. calcula o pior F1 de cada configuração nos pacientes **sorteados**;
2. escolhe a melhor delas;
3. mede essa escolhida nos pacientes que **ficaram de fora** do sorteio (~37% dos 336).

A média do passo 3 estima o desempenho do procedimento inteiro, "testar várias e ficar com a melhor", e não o da vencedora com
sorte. Os sorteios são os mesmos 2.000 do `05_eval` (estratificados: cada classe mantém o seu n).

In [9]:
# ---- cópias do 05_eval: matriz de confusão e painel, vetorizados nos sorteios do bootstrap ----
GS = CLASSES.index("GS")
SEMENTES = [SEED, 142, 242, 342, 442]


def confusoes(y_, yp, idx=None):
    # matrizes de confusão (linha = real, coluna = previsto); com idx (B, n), uma por reamostragem, sem laço
    idx = np.arange(len(y_))[None] if idx is None else idx
    cod = y_[idx] * 4 + yp[idx] + 16 * np.arange(len(idx))[:, None]
    return np.bincount(cod.ravel(), minlength=16 * len(idx)).reshape(-1, 4, 4)


def painel(cm):
    # cm (..., 4, 4) → métricas com forma (...); MCC multiclasse de Gorodkin, o mesmo do scikit-learn
    vp = np.diagonal(cm, axis1=-2, axis2=-1)
    real, prev = cm.sum(-1), cm.sum(-2)
    s, c = cm.sum((-2, -1)), vp.sum(-1)
    f1 = 2 * vp / np.maximum(real + prev, 1)
    mcc = (c * s - (prev * real).sum(-1)) / np.sqrt((s**2 - (prev**2).sum(-1)) * (s**2 - (real**2).sum(-1)))
    return {"MCC": mcc, "acurácia balanceada": (vp / real).mean(-1), "F1 macro": f1.mean(-1), "pior F1": f1.min(-1),
            **{f"F1 {k}": f1[..., i] for i, k in enumerate(CLASSES)}}


rng = np.random.default_rng(SEED)  # os mesmos 2.000 sorteios estratificados do 05_eval
idx = np.concatenate([rng.choice(np.flatnonzero(y == c), (2000, (y == c).sum())) for c in range(4)], axis=1)


def bbc_cv(preds):
    # BBC-CV: em cada sorteio, escolhe a configuração de maior pior F1 nos pacientes sorteados e mede essa escolha nos que
    # ficaram de fora. preds: {nome: classe prevista (336,)}
    vezes = np.stack([np.bincount(i, minlength=len(y)) for i in idx])  # (2.000, 336): vezes que cada paciente saiu
    dentro, fora = [], []
    for yp in preds.values():
        celula = np.eye(16, dtype=int)[y * 4 + yp]                        # (336, 16): célula da matriz de cada paciente
        dentro.append(painel((vezes @ celula).reshape(-1, 4, 4))["pior F1"])
        fora.append(painel(((vezes == 0) @ celula).reshape(-1, 4, 4))["pior F1"])
    escolha = np.argmax(dentro, axis=0)
    v = np.asarray(fora)[escolha, np.arange(len(idx))]
    return {"maior pior F1 (ingênuo)": max(float(painel(confusoes(y, yp)[0])["pior F1"]) for yp in preds.values()),
            "BBC-CV": v.mean(), "IC 2,5%": np.percentile(v, 2.5), "IC 97,5%": np.percentile(v, 97.5),
            "mais escolhida": list(preds)[np.bincount(escolha).argmax()],
            "% dos sorteios": 100 * np.bincount(escolha).max() / len(idx)}


# autoteste 1: a matriz pela contagem de sorteios é a mesma do confusoes() do 05_eval
rt = np.random.default_rng(0)
yp_t = np.where(rt.random(len(y)) < 0.85, y, rt.integers(0, 4, len(y)))
vezes = np.stack([np.bincount(i, minlength=len(y)) for i in idx[:50]])
assert np.array_equal((vezes @ np.eye(16, dtype=int)[y * 4 + yp_t]).reshape(-1, 4, 4), confusoes(y, yp_t, idx[:50]))
# autoteste 2: 20 configurações de mesma qualidade (85% de acerto ao acaso): a maior é otimista, a BBC-CV não
ruido = {i: np.where(rt.random(len(y)) < 0.85, y, rt.integers(0, 4, len(y))) for i in range(20)}
r = bbc_cv(ruido)
media = np.mean([painel(confusoes(y, p)[0])["pior F1"] for p in ruido.values()])
assert r["maior pior F1 (ingênuo)"] > media + 0.04 and abs(r["BBC-CV"] - media) < 0.04, (r, media)
print(f"20 configurações iguais: média {media:.3f} · maior {r['maior pior F1 (ingênuo)']:.3f} · BBC-CV {r['BBC-CV']:.3f}")

20 configurações iguais: média 0.790 · maior 0.877 · BBC-CV 0.787


### As variantes, em código

Uma função de treino só, `treinar_variante()`: é o `treinar_cnn()` de cima, com as mudanças ligáveis. Com tudo desligado, é a
Small CNN atual.

- **Ramo largo** (*wide & deep*; relatório, seção 4.1): uma camada densa direto sobre os 53.466 valores da imagem (133 × 134
  pixels × 3 canais), somada aos logits da Small CNN. É uma logística dentro da rede: cada gene e canal ganha o seu peso, e o
  pixel do MLH1 volta a ter identidade. O L2 de 1/(2n) reproduz o C = 1 da logística de cima. No scikit-learn, a penalidade é
  ‖w‖²/2 somada à perda **total**; dividindo tudo pelo n, fica ‖w‖²/(2n) somada à perda **média**, que é a do Keras.
- **AdamW** (relatório, seção 5.7): o Adam com o decaimento de peso separado do gradiente. No Keras, a cada passo cada peso perde
  lr × wd = 0,001 × 1,0 = 0,1% do valor. Em 480 passos (60 épocas de ~8 lotes), sem gradiente nenhum, sobrariam 62%. Bias e
  BatchNorm não decaem.
- **Focal loss** (LIN et al., 2018; relatório, seção 5.2): multiplica a perda de cada paciente por (1 − p)^γ, com p a
  probabilidade da classe certa. Com γ = 2, um paciente com p = 0,9 passa a pesar 1% do que pesava, e um com p = 0,4, 36%.
- **Group DRO** (SAGAWA et al., 2020; relatório, seção 5.4), com grupo = classe. A cada lote, o peso q de cada classe é
  multiplicado por exp(η × perda média da classe no lote) e renormalizado, e a perda do lote é Σ q × perda da classe. A classe
  que vai pior ganha peso aos poucos. Substitui o peso de classe fixo. Pelo relatório, o group DRO só funciona com regularização
  forte, por isso ele entra junto com o AdamW.

As duas conferências no fim da célula: o group DRO num lote de 4 pacientes feito à mão, e 1 época de cada variante em 40
pacientes, para que um erro de código apareça em segundos, e não depois de horas de treino.

In [10]:
def rede(entrada, largo=False, profundo=True, l2=0.0, achatar=False):
    # wide & deep: logits da Small CNN + logits de uma densa sobre todos os pixels (o ramo largo é uma logística)
    x = keras.Input(entrada)
    ramos = [small_cnn(entrada, achatar=achatar)(x)] if profundo else []
    if largo:
        ramos.append(layers.Dense(len(CLASSES), kernel_regularizer=keras.regularizers.L2(l2))(layers.Flatten()(x)))
    return keras.Model(x, ramos[0] if len(ramos) == 1 else layers.Add()(ramos))


def focal(y_true, logits):
    # focal loss com γ = 2 e α = 1 (o α padrão do Keras, 0,25, só mudaria a escala da perda contra o L2)
    um = tf.one_hot(tf.cast(tf.reshape(y_true, [-1]), tf.int32), len(CLASSES))
    return keras.losses.categorical_focal_crossentropy(um, logits, alpha=1.0, gamma=2.0, from_logits=True)


class GroupDRO(keras.losses.Loss):
    # group DRO online, grupo = classe: q_g ← q_g · exp(η · perda média da classe g no lote); perda = Σ_g q_g · perda_g
    def __init__(self, eta=0.01):
        super().__init__(reduction="sum")
        self.eta = eta
        self.q = tf.Variable(tf.fill([len(CLASSES)], 1 / len(CLASSES)), trainable=False)

    def call(self, y_true, logits):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        ce = tf.nn.sparse_softmax_cross_entropy_with_logits(y_true, logits)
        g = tf.one_hot(y_true, len(CLASSES))                # (lote, 4)
        n = tf.maximum(tf.reduce_sum(g, 0), 1.0)
        perda_g = tf.reduce_sum(g * ce[:, None], 0) / n     # classe ausente do lote: perda 0, q dela não sobe
        q = self.q.assign(self.q * tf.exp(self.eta * tf.stop_gradient(perda_g)))
        q = self.q.assign(q / tf.reduce_sum(q))
        return ce * tf.reduce_sum(g * q / n, 1)             # somado no lote = Σ_g q_g · perda_g


def treinar_variante(x_tr, y_tr, x_te, seed, largo=False, profundo=True, adamw=False, perda="ce", epocas=60, achatar=False):
    # = treinar_cnn(), com as mudanças pré-registradas; com tudo desligado, é a Small CNN atual
    keras.backend.clear_session()
    gc.collect()
    tr, va = next(StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed).split(x_tr, y_tr))
    keras.utils.set_random_seed(seed)
    modelo = rede(x_tr.shape[1:], largo, profundo, l2=1 / (2 * len(tr)), achatar=achatar)  # L2 do ramo largo = C 1 da logística
    otimizador = keras.optimizers.AdamW(1e-3, weight_decay=1.0) if adamw else keras.optimizers.Adam(1e-3)
    if adamw:
        otimizador.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])  # só os kernels decaem
    funcao = (GroupDRO() if perda == "dro" else focal if perda == "focal"
              else keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    modelo.compile(otimizador, funcao)
    f1_val = F1MacroVal(x_tr[va], y_tr[va])
    parada = keras.callbacks.EarlyStopping("val_f1_macro", mode="max", patience=15, restore_best_weights=True)
    pesos = None if perda == "dro" else dict(enumerate(pesos_por_classe(y_tr[tr])))  # o DRO substitui o peso de classe
    modelo.fit(x_tr[tr], y_tr[tr], epochs=epocas, batch_size=32, class_weight=pesos, callbacks=[f1_val, parada], verbose=0)
    log(f"  {len(f1_val.f1s)} épocas, melhor F1 macro na validação interna {max(f1_val.f1s):.3f}")
    global MELHOR_EPOCA  # para o log de acompanhamento
    MELHOR_EPOCA = int(np.argmax(f1_val.f1s)) + 1
    return softmax(modelo.predict(x_te, verbose=0), axis=1)


OOF_O = RESULTS / "oof_otim"  # uma previsão por variante, semente e fold; apague a pasta para treinar de novo
OOF_O.mkdir(exist_ok=True)
FORMATOS = {"hilbert": lambda z: np.moveaxis(para_hilbert(z), 1, -1), "vetor": lambda z: np.moveaxis(z, 1, -1)}
ITEM2 = {"largo": {"largo": True, "profundo": False}, "largo_profundo": {"largo": True}}
ITEM4 = {"adamw": {"adamw": True}, "adamw_focal": {"adamw": True, "perda": "focal"},
         "adamw_dro": {"adamw": True, "perda": "dro"}}


def gravados(modelo, s):
    # molde do arquivo de uma semente já treinada da Small CNN atual: 04_models (semente 42) ou bloco B do 05_eval
    return str(OOF / f"{modelo}_fold{{k}}.csv") if s == SEED else str(RESULTS / "oof_b" / f"{modelo}_s{s}_fold{{k}}.csv")


def acomp_base(rotulo, te, yp, semente):
    # log de acompanhamento: pior F1 da rede base (profundo + adamw_dro) nos mesmos pacientes e na mesma semente, e a diferença
    arqs = [OOF_O / f"profundo_adamw_dro_s{semente}_fold{k}.csv" for k in range(K)]
    if all(a.exists() for a in arqs):  # num treino do zero, a rede base pode ainda não existir
        b = pd.concat(map(pd.read_csv, arqs)).set_index("amostra").loc[coorte.amostra, PROBS].to_numpy().argmax(axis=1)
        pior_b = f1_cm(matriz(y[te], b[te])).min()
        acomp(f"  {rotulo}: pior {virgula(pior_b)}  →  {virgula(f1_cm(matriz(y[te], yp)).min() - pior_b, True)}")


def rodar_o(nome, semente, cfg, formato="hilbert", molde=None, preparar=None, pasta=OOF_O):
    # previsões fora do fold (336, 4) na ordem do coorte_cv.csv; reaproveita o que já existe e treina o resto
    # preparar(k): outra entrada, já padronizada com o treino do fold k (testes R da PARTE C); sem ela, a entrada atual
    proba, treinou = np.empty((len(y), len(CLASSES))), False
    for k in range(K):
        te = fold == k
        arq = Path(molde.format(k=k)) if molde and Path(molde.format(k=k)).exists() else pasta / f"{nome}_s{semente}_fold{k}.csv"
        if not arq.exists():
            log(f"{nome} · semente {semente} · fold {k}")
            acomp(f"{nome} · semente {semente} · fold {k}")
            x = FORMATOS[formato](preparar(k) if preparar else padronizar(tensor, padronizacao[k]))
            p = treinar_variante(x[~te], y[~te], x[te], semente + k, **cfg)
            pd.DataFrame(p, columns=PROBS).assign(amostra=coorte.amostra[te].to_numpy()).to_csv(arq, index=False)
            del x
            treinou, yp = True, p.argmax(axis=1)
            cm = matriz(y[te], yp)
            pior, por_classe = texto_f1(cm)
            acomp(f"  fold {k} terminado (melhor época {MELHOR_EPOCA})")
            acomp(f"  TESTE do fold: {pior} · {por_classe} · CIN→GS {cm[0, 2]} de {cm[0].sum()} · GS→CIN {cm[2, 0]} de {cm[2].sum()}")
            acomp_base("rede base neste fold", te, yp, semente)
        proba[te] = pd.read_csv(arq).set_index("amostra").loc[coorte.amostra[te], PROBS].to_numpy()
    if treinou:
        yp = proba.argmax(axis=1)
        cm = matriz(y, yp)
        pior, por_classe = texto_f1(cm)
        acomp(f"{nome} · semente {semente} terminada ({len(y)} pacientes)")
        acomp(f"  {pior} · {por_classe} · CIN→GS {cm[0, 2]},0 · GS→CIN {cm[2, 0]},0")
        acomp_base("rede base, mesma semente", np.ones(len(y), bool), yp, semente)
    return proba


# autoteste do group DRO: num lote em que a classe 1 vai mal, o peso dela sobe, e a perda é Σ q_g · perda_g
dro = GroupDRO(eta=1.0)
yt, lg = tf.constant([0, 0, 1, 1]), tf.constant([[3.0, 0, 0, 0]] * 4)
total, q = float(dro(yt, lg)), dro.q.numpy()
ce = tf.nn.sparse_softmax_cross_entropy_with_logits(yt, lg).numpy()
assert q[1] > q[0] > q[2] == q[3] and np.isclose(q.sum(), 1)
assert np.isclose(total, q[0] * ce[:2].mean() + q[1] * ce[2:].mean(), rtol=1e-5)

# autoteste das variantes: 1 época em 40 pacientes de treino do fold 0, para pegar erro antes das horas de treino
sub = np.concatenate([np.flatnonzero((y == c) & (fold != 0))[:10] for c in range(4)])
x = FORMATOS["hilbert"](padronizar(tensor[sub], padronizacao[0]))
for nome, cfg in {"profundo": {}, **ITEM2, **ITEM4, "largo_profundo + adamw_dro": ITEM2["largo_profundo"] | ITEM4["adamw_dro"]}.items():
    p = treinar_variante(x, y[sub], x[:3], SEED, **cfg, epocas=1)
    assert p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4), nome
del x
print({nome: rede((*FORMA, 3), **{k: v for k, v in cfg.items() if k in ("largo", "profundo")}).count_params()
       for nome, cfg in {"profundo": {}, **ITEM2}.items()})

22:41:42 [RAM 1088 MB, pico 1225 MB]   1 épocas, melhor F1 macro na validação interna 0.071


22:41:53 [RAM 993 MB, pico 1225 MB]   1 épocas, melhor F1 macro na validação interna 0.071


22:42:22 [RAM 1208 MB, pico 1305 MB]   1 épocas, melhor F1 macro na validação interna 0.071


22:43:01 [RAM 1209 MB, pico 1345 MB]   1 épocas, melhor F1 macro na validação interna 0.071


22:43:42 [RAM 1265 MB, pico 1348 MB]   1 épocas, melhor F1 macro na validação interna 0.071


22:44:05 [RAM 1364 MB, pico 1481 MB]   1 épocas, melhor F1 macro na validação interna 0.071


22:44:27 [RAM 1245 MB, pico 1516 MB]   1 épocas, melhor F1 macro na validação interna 0.071


{'profundo': 61476, 'largo': 213868, 'largo_profundo': 275344}


### Referência estável: as 5 sementes que já existem

Antes de qualquer variante, a Small CNN (Hilbert) atual (`profundo`) e a Small CNN (vetor 1D), com as 5 sementes já gravadas e
sem treino. Duas formas de resumir: a **média do pior F1** das 5 execuções (a métrica pré-registrada) e o pior F1 do **ensemble**
(a média das probabilidades das 5, uma previsão só). Embaixo, a BBC-CV nas 10 execuções: quanto vale, de verdade, "ficar com a
melhor semente".

In [11]:
PO = {}  # (variante, semente) → probabilidades fora do fold (336, 4)
for s in SEMENTES:
    PO[("profundo", s)] = rodar_o("profundo", s, {}, molde=gravados("cnn_hilbert", s))
    PO[("vetor 1D", s)] = rodar_o("vetor", s, {}, "vetor", gravados("cnn_vetor", s))
P_LOG = pred[pred.modelo == "logistica"].set_index("amostra").loc[coorte.amostra, PROBS].to_numpy()


def resumo(variantes):
    # por variante: média e desvio das 5 sementes e o pior F1 do ensemble; a logística (determinística) como referência
    linhas = {}
    for v in variantes:
        ps = [PO[(v, s)] for s in SEMENTES]
        m = pd.DataFrame([{k: float(x) for k, x in painel(confusoes(y, p.argmax(axis=1))[0]).items()} for p in ps])
        linhas[v] = {"pior F1": m["pior F1"].mean(), "dp": m["pior F1"].std(), "mín.": m["pior F1"].min(),
                     "máx.": m["pior F1"].max(), **{f"F1 {c}": m[f"F1 {c}"].mean() for c in CLASSES}, "MCC": m.MCC.mean(),
                     "AUC-PR GS": np.mean([average_precision_score(y == GS, p[:, GS]) for p in ps]),
                     "pior F1 do ensemble": float(painel(confusoes(y, np.mean(ps, axis=0).argmax(axis=1))[0])["pior F1"])}
    m = {k: float(x) for k, x in painel(confusoes(y, P_LOG.argmax(axis=1))[0]).items()}
    linhas["logística (1 execução)"] = {"pior F1": m["pior F1"], **{f"F1 {c}": m[f"F1 {c}"] for c in CLASSES},
                                       "MCC": m["MCC"], "AUC-PR GS": average_precision_score(y == GS, P_LOG[:, GS])}
    return pd.DataFrame(linhas).T


display(resumo(["profundo", "vetor 1D"]).round(3))
pd.Series(bbc_cv({f"{v} s{s}": PO[(v, s)].argmax(axis=1) for v in ["profundo", "vetor 1D"] for s in SEMENTES}),
          name="ficar com a melhor das 10 execuções").to_frame()

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble
profundo,0.688,0.025,0.660,0.722,0.919,0.977,0.688,0.892,0.821,0.748,0.772
vetor 1D,0.712,0.016,0.695,0.737,0.909,0.978,0.712,0.892,0.823,0.772,0.730
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN


,ficar com a melhor das 10 execuções
maior pior F1 (ingênuo),0.736842
BBC-CV,0.695636
"IC 2,5%",0.551724
"IC 97,5%",0.818182
mais escolhida,vetor 1D s442
% dos sorteios,46.85


## Item 2 — ramo linear ao lado da Small CNN (*wide & deep*)

Duas variantes novas, 5 sementes cada (o `profundo` já está acima, sem retreino):
- `largo`: só o ramo linear, sem convolução. É o controle: uma logística treinada como a rede (Adam, parada antecipada, peso de
  classe), para separar o efeito do ramo do efeito do treino;
- `largo_profundo`: os dois ramos somados nos logits.

Como ler: se `largo_profundo` > `profundo`, o peso por pixel ajuda a Small CNN. Se `largo_profundo` ≈ `largo`, quem trabalha é o
ramo linear, e a convolução não acrescenta nada. É o que pergunta a pendência "a imagem é necessária?".

In [12]:
t0 = time.time()
for v, cfg in ITEM2.items():
    for s in SEMENTES:
        log(f"item 2: {v}, semente {s}")
        PO[(v, s)] = rodar_o(v, s, cfg)
log(f"item 2 terminado em {(time.time() - t0) / 60:.0f} min")
resumo(["profundo", *ITEM2]).round(3)

22:44:32 [RAM 1250 MB, pico 1516 MB] item 2: largo, semente 42


22:44:32 [RAM 1250 MB, pico 1516 MB] item 2: largo, semente 142


22:44:33 [RAM 1250 MB, pico 1516 MB] item 2: largo, semente 242
22:44:33 [RAM 1250 MB, pico 1516 MB] item 2: largo, semente 342


22:44:33 [RAM 1250 MB, pico 1516 MB] item 2: largo, semente 442
22:44:33 [RAM 1250 MB, pico 1516 MB] item 2: largo_profundo, semente 42


22:44:33 [RAM 1250 MB, pico 1516 MB] item 2: largo_profundo, semente 142


22:44:34 [RAM 1250 MB, pico 1516 MB] item 2: largo_profundo, semente 242


22:44:34 [RAM 1250 MB, pico 1516 MB] item 2: largo_profundo, semente 342
22:44:34 [RAM 1250 MB, pico 1516 MB] item 2: largo_profundo, semente 442


22:44:34 [RAM 1250 MB, pico 1516 MB] item 2 terminado em 0 min


,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble
profundo,0.688,0.025,0.660,0.722,0.919,0.977,0.688,0.892,0.821,0.748,0.772
largo,0.710,0.051,0.642,0.760,0.911,0.933,0.710,0.921,0.807,0.710,0.714
largo_profundo,0.667,0.070,0.558,0.733,0.909,0.922,0.667,0.876,0.787,0.686,0.701
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN


## Item 4 — perda e regularização para o pior caso

Sobre a melhor entre `profundo` e `largo_profundo` (pré-registrado: maior média do pior F1; em empate, `profundo`). O `largo` não
entra: ele não tem convolução, e a pergunta é otimizar a Small CNN. Três variantes, 5 sementes cada:
- `adamw`: só o decaimento de peso forte. É a referência das outras duas;
- `adamw_focal`: o mesmo, com focal loss;
- `adamw_dro`: o mesmo, com group DRO.

Como ler: `adamw` contra a base mede a regularização; `adamw_focal` e `adamw_dro` contra `adamw` medem a perda.

In [13]:
medias = {v: np.mean([painel(confusoes(y, PO[(v, s)].argmax(axis=1))[0])["pior F1"] for s in SEMENTES])
          for v in ["profundo", "largo_profundo"]}
BASE4 = "largo_profundo" if medias["largo_profundo"] > medias["profundo"] else "profundo"  # em empate, profundo
log(f"item 4 sobre {BASE4} (pior F1 médio: profundo {medias['profundo']:.3f}, largo_profundo {medias['largo_profundo']:.3f})")
t0 = time.time()
for v, cfg in ITEM4.items():
    for s in SEMENTES:
        log(f"item 4: {BASE4} + {v}, semente {s}")
        PO[(f"{BASE4} + {v}", s)] = rodar_o(f"{BASE4}_{v}", s, ITEM2.get(BASE4, {}) | cfg)
log(f"item 4 terminado em {(time.time() - t0) / 60:.0f} min")
resumo([BASE4, *(f"{BASE4} + {v}" for v in ITEM4)]).round(3)

22:44:34 [RAM 1250 MB, pico 1516 MB] item 4 sobre profundo (pior F1 médio: profundo 0.688, largo_profundo 0.667)
22:44:34 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw, semente 42


22:44:35 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw, semente 142
22:44:35 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw, semente 242


22:44:35 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw, semente 342
22:44:35 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw, semente 442


22:44:35 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw_focal, semente 42


22:44:36 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw_focal, semente 142


22:44:36 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw_focal, semente 242


... (3 outputs left out)

22:44:37 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw_dro, semente 142


22:44:37 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw_dro, semente 242
22:44:37 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw_dro, semente 342


22:44:38 [RAM 1250 MB, pico 1516 MB] item 4: profundo + adamw_dro, semente 442
22:44:38 [RAM 1250 MB, pico 1516 MB] item 4 terminado em 0 min


,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble
profundo,0.688,0.025,0.660,0.722,0.919,0.977,0.688,0.892,0.821,0.748,0.772
profundo + adamw,0.691,0.022,0.661,0.714,0.919,0.970,0.691,0.878,0.816,0.776,0.750
profundo + adamw_focal,0.692,0.011,0.682,0.710,0.914,0.969,0.692,0.932,0.821,0.739,0.740
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN


## Comparação final

Todas as variantes pré-registradas, nas 336 previsões fora do fold:
1. o resumo de cada uma (média das 5 sementes e ensemble), gravado em `results/04_models_otimizacao.csv`;
2. a diferença pareada contra o `profundo` (mesma semente) e contra a logística, como média das 5 sementes, com IC 95% pelo
   bootstrap pareado (os mesmos 2.000 sorteios do `05_eval`);
3. a BBC-CV sobre as 6 configurações (o ensemble de cada uma) e sobre as 30 execuções: o valor honesto de "ficar com a melhor".

In [14]:
VARIANTES = list(dict.fromkeys(v for v, _ in PO if v != "vetor 1D"))
tab_o = resumo(VARIANTES)
tab_o.to_csv(RESULTS / "04_models_otimizacao.csv")
display(tab_o.round(3))


def dif_media(pares, met):
    # média, sobre as sementes, da diferença a − b; IC 95% pelo bootstrap pareado (cópia do 05_eval, bloco D)
    pont = np.mean([painel(confusoes(y, a)[0])[met] - painel(confusoes(y, b)[0])[met] for a, b in pares])
    lo, hi = np.percentile(np.mean([painel(confusoes(y, a, idx))[met] - painel(confusoes(y, b, idx))[met]
                                    for a, b in pares], axis=0), [2.5, 97.5])
    return f"{pont:+.3f} ({lo:+.3f} a {hi:+.3f})"


MET = ["pior F1", "F1 GS", "F1 EBV", "MCC"]
comparacoes = {}
for v in VARIANTES:
    a = [PO[(v, s)].argmax(axis=1) for s in SEMENTES]
    if v != "profundo":
        b = [PO[("profundo", s)].argmax(axis=1) for s in SEMENTES]
        comparacoes[f"{v} − profundo"] = {m: dif_media(list(zip(a, b)), m) for m in MET}
    comparacoes[f"{v} − logística"] = {m: dif_media([(p, P_LOG.argmax(axis=1)) for p in a], m) for m in MET}
print("Diferença (média das 5 sementes) e IC 95%:")
display(pd.DataFrame(comparacoes).T)

ens = {v: np.mean([PO[(v, s)] for s in SEMENTES], axis=0).argmax(axis=1) for v in VARIANTES}
pd.DataFrame({"6 configurações (ensemble de cada)": bbc_cv(ens),
              "30 execuções": bbc_cv({f"{v} s{s}": PO[(v, s)].argmax(axis=1) for v in VARIANTES for s in SEMENTES})})

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble
profundo,0.688,0.025,0.660,0.722,0.919,0.977,0.688,0.892,0.821,0.748,0.772
largo,0.710,0.051,0.642,0.760,0.911,0.933,0.710,0.921,0.807,0.710,0.714
largo_profundo,0.667,0.070,0.558,0.733,0.909,0.922,0.667,0.876,0.787,0.686,0.701
profundo + adamw,0.691,0.022,0.661,0.714,0.919,0.970,0.691,0.878,0.816,0.776,0.750
profundo + adamw_focal,0.692,0.011,0.682,0.710,0.914,0.969,0.692,0.932,0.821,0.739,0.740
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN


Diferença (média das 5 sementes) e IC 95%:


,pior F1,F1 GS,F1 EBV,MCC
profundo − logística,-0.062 (-0.114 a -0.011),-0.062 (-0.114 a -0.011),-0.059 (-0.105 a -0.019),-0.022 (-0.055 a +0.011)
largo − profundo,+0.022 (-0.023 a +0.068),+0.022 (-0.023 a +0.068),+0.029 (-0.014 a +0.072),-0.014 (-0.046 a +0.017)
largo − logística,-0.040 (-0.087 a +0.002),-0.040 (-0.087 a +0.002),-0.030 (-0.054 a -0.012),-0.035 (-0.060 a -0.011)
largo_profundo − profundo,-0.021 (-0.080 a +0.037),-0.021 (-0.080 a +0.037),-0.016 (-0.054 a +0.033),-0.034 (-0.069 a +0.001)
largo_profundo − logística,-0.083 (-0.144 a -0.033),-0.083 (-0.144 a -0.033),-0.074 (-0.112 a -0.040),-0.055 (-0.085 a -0.028)
profundo + adamw − profundo,+0.003 (-0.026 a +0.032),+0.003 (-0.026 a +0.032),-0.014 (-0.043 a +0.011),-0.005 (-0.019 a +0.010)
profundo + adamw − logística,-0.059 (-0.109 a -0.011),-0.059 (-0.109 a -0.011),-0.073 (-0.135 a -0.018),-0.026 (-0.061 a +0.007)
profundo + adamw_focal − profundo,+0.004 (-0.016 a +0.026),+0.004 (-0.016 a +0.026),+0.040 (+0.011 a +0.073),+0.000 (-0.013 a +0.014)
profundo + adamw_focal − logística,-0.058 (-0.111 a -0.008),-0.058 (-0.111 a -0.008),-0.019 (-0.060 a +0.018),-0.021 (-0.053 a +0.011)
profundo + adamw_dro − profundo,+0.030 (+0.003 a +0.058),+0.030 (+0.002 a +0.058),+0.030 (-0.005 a +0.067),+0.013 (-0.005 a +0.031)


,6 configurações (ensemble de cada),30 execuções
maior pior F1 (ingênuo),0.772277,0.76
BBC-CV,0.751472,0.704823
"IC 2,5%",0.604535,0.555556
"IC 97,5%",0.869565,0.823529
mais escolhida,profundo,largo s242
% dos sorteios,80.25,36.05


### Leitura da otimização

Pré-registro no diário (2026-10-31). Métrica principal: pior F1 no argmax, média das 5 sementes, nas 336 previsões fora do
fold (GS = 46, EBV = 29). Em todas as variantes, e em todas as sementes, a classe que cai é o **GS**: o pior F1 é o F1 do GS.

| variante | pior F1 (média ± dp) | MCC | AUC-PR GS |
|---|---|---|---|
| `profundo` (Small CNN (Hilbert) atual) | 0,688 ± 0,025 | 0,821 | 0,748 |
| `largo` (só o ramo linear, sem convolução) | 0,710 ± 0,051 | 0,807 | 0,710 |
| `largo_profundo` (wide & deep) | 0,667 ± 0,070 | 0,787 | 0,686 |
| `profundo` + AdamW | 0,691 ± 0,022 | 0,816 | 0,776 |
| `profundo` + AdamW + focal loss | 0,692 ± 0,011 | 0,821 | 0,739 |
| `profundo` + AdamW + group DRO | **0,717 ± 0,015** | 0,834 | 0,778 |
| logística (1 execução) | 0,750 | 0,842 | 0,788 |

1. **O wide & deep não ajudou.** O `largo_profundo` ficou abaixo do `profundo` (−0,021, IC −0,080 a +0,037), com o maior dp.
   Pela regra pré-registrada, a base do item 4 foi o `profundo`.
2. **A melhor variante é AdamW + group DRO** (0,717 ± 0,015):
   - contra o `profundo`: +0,030 (IC +0,003 a +0,058). O IC exclui o 0 por pouco, e sem correção para as 5 comparações
     contra o `profundo`. No MCC, +0,013 (IC −0,005 a +0,031);
   - contra a logística: −0,033 (IC −0,081 a +0,013). Não a alcança;
   - AdamW sozinho (+0,003) e com focal loss (+0,004) não mudam o pior F1. A focal loss sobe o EBV (+0,040, IC +0,011 a
     +0,073), mas o GS fica igual.
3. **BBC-CV** (corrige o otimismo de ficar com a melhor):
   - nas 30 execuções: a melhor execução isolada dá 0,760; corrigida, **0,705** (IC 0,556 a 0,824). A mais escolhida nos
     sorteios é uma semente do `largo` (36%), não o DRO;
   - nas 6 configurações (média das probabilidades das 5 sementes): 0,772 → **0,751** (IC 0,605 a 0,870). A mais escolhida
     é o `profundo` (80%), porque o ensemble do DRO (0,757) fica abaixo do ensemble do `profundo` (0,772);
   - leitura: o "ganho" de escolher a melhor execução some com a correção (0,705 está na faixa das variantes). O ganho do
     DRO na média das sementes não aparece no ensemble. O 0,751 corrigido empata com a logística (0,750), mas compara um
     ensemble de 5 redes com uma execução da logística, e não é a métrica principal do pré-registro.
4. **A imagem é necessária?** O `largo` é uma camada densa sobre os mesmos 53.466 valores da imagem, sem convolução. Ele
   fica +0,022 acima do `profundo` (IC −0,023 a +0,068): a convolução não acrescenta nada mensurável a um modelo linear nos
   mesmos valores. E nenhuma variante com convolução passa da logística. O `largo` é o análogo da logística treinado como a
   rede (Adam, parada antecipada), e fica abaixo dela (−0,040, IC −0,087 a +0,002). Tudo vai na direção do D2 (permutação
   gene → pixel sem efeito) e do bloco B do `05_eval`. O critério formal de descarte (BBC-CV pareada "melhor imagem − melhor
   tabular", PARTE C de `docs/relatorio_imagem.txt`) ainda não foi rodado.
5. **Cautela:** com GS = 46, um GS a mais ou a menos acertado muda o F1 do GS em 0,01 a 0,02. Diferenças de 0,02 a 0,05
   entre as variantes costumam estar no ruído. Só a do DRO contra o `profundo` sai do 0, e por pouco.

# Testes com rede da PARTE C (2026-11-01)

Plano em `docs/relatorio_imagem.txt` (PARTE C); pré-registro no diário (2026-11-01, "Pré-registro do R1a e do R1b"). Decisão do
usuário (2026-11-01): todo treino de modelo entra neste notebook, e a avaliação final (BBC-par contra a melhor tabular) fica no
`05_eval`, que lê as previsões gravadas em `results/oof_r/`.

Rede base de todos os R: `profundo + adamw_dro` (pior F1 0,717 ± 0,015; decisão do usuário, 2026-10-31), 5 sementes, os mesmos
folds, argmax.

## R1 — CNV com zona morta

- **R1a (3 canais):** na CNV bruta, |log2| < 0,1 vira 0, e o resto fica como está (o 0,1 não é subtraído).
- **R1b (4 canais):** a mesma zona morta, e a CNV separada em ganho = max(log2, 0) e perda = max(−log2, 0). Canais: mutação,
  ganho, perda, metilação.
- Nas duas: mutação em `log1p`, como hoje; z por canal com a média e o desvio do treino de cada fold (ignorando `NaN`); `NaN` → 0
  depois. A zona morta e a separação vêm antes do z.
- **Hipótese** (PARTE C, R1): sem os valores perto de 0 e com ganho e perda separados, a média global de um filtro passa a medir a
  fração do genoma alterada, que é a variável que separa o GS do CIN.
- O limiar 0,1 vem do P1 (bloco E do `05_eval`): com ele, a fração alterada separa os GS dos CIN que a rede chamou de GS com AUC
  0,814, contra 0,691 com 0,2.
- A logística com a mesma entrada roda antes da rede. Dif(X) = pior F1 da rede com a entrada X − pior F1 da logística com X.

In [15]:
OOF_R = RESULTS / "oof_r"  # previsões dos testes R; apague um arquivo para treinar aquele fold de novo
OOF_R.mkdir(exist_ok=True)
BASE = ITEM4["adamw_dro"]  # rede base dos R: profundo + adamw_dro (decisão do usuário, 2026-10-31)
ZONA = 0.1                  # |log2| < 0,1 vira 0 (P1, bloco E do 05_eval)


def entrada_r1(x, quatro):
    # valores antes do z: mutação em log1p; CNV com zona morta e, se quatro, separada em ganho e perda; NaN continua NaN
    x = transformar(x)
    with np.errstate(invalid="ignore"):
        cnv = np.where(np.abs(x[:, 1]) < ZONA, 0, x[:, 1]).astype(np.float32)
    canais = [x[:, 0], np.maximum(cnv, 0), np.maximum(-cnv, 0), x[:, 2]] if quatro else [x[:, 0], cnv, x[:, 2]]
    return np.stack(canais, axis=1)


def preparar_r1(k, quatro):
    # z por canal com a média e o desvio do treino do fold k (ignorando NaN); NaN → 0 depois, como em padronizar()
    x = entrada_r1(tensor, quatro)
    media = np.nanmean(x[fold != k], axis=(0, 2), keepdims=True)
    desvio = np.nanstd(x[fold != k], axis=(0, 2), keepdims=True)
    return np.nan_to_num((x - media) / desvio, nan=0.0)


# autoteste, antes das horas de treino
a, b = entrada_r1(tensor, False), entrada_r1(tensor, True)
cnv = a[:, 1]
assert (np.isnan(cnv) == np.isnan(tensor[:, 1])).all()                    # a zona morta não cria nem apaga NaN
assert not ((np.abs(np.nan_to_num(cnv)) > 0) & (np.abs(np.nan_to_num(cnv)) < ZONA)).any()  # nada entre 0 e 0,1
assert np.allclose(b[:, 1] - b[:, 2], cnv, equal_nan=True)                 # ganho − perda = CNV
assert not ((b[:, 1] > 0) & (b[:, 2] > 0)).any()                           # nunca ganho e perda no mesmo gene
bruta = tensor[:, 1][~np.isnan(tensor[:, 1])]
print(f"CNV zerada pela zona morta: {(np.abs(bruta) < ZONA).mean():.1%} dos valores (com |log2| < 0,2: {(np.abs(bruta) < 0.2).mean():.1%})")
del a, b, cnv, bruta
for k in range(K):
    atual, r1a, r1b = padronizar(tensor, padronizacao[k]), preparar_r1(k, False), preparar_r1(k, True)
    assert np.allclose(r1a[:, [0, 2]], atual[:, [0, 2]], atol=1e-4), k       # mutação e metilação iguais às de hoje
    assert np.allclose(r1b[:, [0, 3]], atual[:, [0, 2]], atol=1e-4), k
    for z in (r1a, r1b):
        assert np.allclose(z[fold != k].mean(axis=(0, 2)), 0, atol=1e-3), k  # média 0 no treino do fold
del atual, r1a, r1b, z
x = FORMATOS["hilbert"](preparar_r1(0, True)[sub])                          # rede com 4 canais: 1 época em 40 pacientes
p = treinar_variante(x, y[sub], x[:3], SEED, **BASE, epocas=1)
assert x.shape[1:] == (*FORMA, 4) and p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4)
del x, p
print({f"{c} canais": rede((*FORMA, c)).count_params() for c in (3, 4)})

CNV zerada pela zona morta: 61.6% dos valores (com |log2| < 0,2: 77.7%)


22:46:37 [RAM 1400 MB, pico 2004 MB]   1 épocas, melhor F1 macro na validação interna 0.071


{'3 canais': 61476, '4 canais': 61620}


In [16]:
R1 = {"R1a": False, "R1b": True}  # nome → 4 canais?
P_LOG_R = {}                       # logística com a mesma entrada (336, 4)
log("=== testes R da PARTE C: R1a e R1b ===")
t0 = time.time()
for nome, quatro in R1.items():
    proba = np.empty((len(y), len(CLASSES)))
    for k in range(K):  # a logística vem antes da rede (C.2)
        arq, te = OOF_R / f"{nome}_logistica_fold{k}.csv", fold == k
        if not arq.exists():
            log(f"{nome} · logística · fold {k}")
            x = preparar_r1(k, quatro).reshape(len(y), -1)
            p = treinar_logistica(x[~te], y[~te], x[te], SEED + k)
            pd.DataFrame(p, columns=PROBS).assign(amostra=coorte.amostra[te].to_numpy()).to_csv(arq, index=False)
            del x
        proba[te] = pd.read_csv(arq).set_index("amostra").loc[coorte.amostra[te], PROBS].to_numpy()
    P_LOG_R[nome] = proba
    for s in SEMENTES:
        log(f"{nome}: semente {s}")
        PO[(nome, s)] = rodar_o(nome, s, BASE, preparar=lambda k, q=quatro: preparar_r1(k, q), pasta=OOF_R)
log(f"R1a e R1b terminados em {(time.time() - t0) / 60:.0f} min")

22:46:40 [RAM 1364 MB, pico 2004 MB] === testes R da PARTE C: R1a e R1b ===


22:46:40 [RAM 1364 MB, pico 2004 MB] R1a: semente 42


22:46:40 [RAM 1364 MB, pico 2004 MB] R1a: semente 142


22:46:41 [RAM 1364 MB, pico 2004 MB] R1a: semente 242


22:46:41 [RAM 1364 MB, pico 2004 MB] R1a: semente 342


22:46:41 [RAM 1364 MB, pico 2004 MB] R1a: semente 442


... (2 outputs left out)

22:46:42 [RAM 1364 MB, pico 2004 MB] R1b: semente 242


22:46:43 [RAM 1364 MB, pico 2004 MB] R1b: semente 342


22:46:43 [RAM 1364 MB, pico 2004 MB] R1b: semente 442


22:46:43 [RAM 1364 MB, pico 2004 MB] R1a e R1b terminados em 0 min


In [17]:
BASE_R = f"{BASE4} + adamw_dro"
CIN = CLASSES.index("CIN")


def cin_gs(p):
    return int(confusoes(y, p.argmax(axis=1))[0][CIN, GS])  # CIN previstos como GS


tab_r = resumo([BASE_R, *R1])
tab_r["CIN → GS"] = [np.mean([cin_gs(PO[(v, s)]) for s in SEMENTES]) for v in [BASE_R, *R1]] + [cin_gs(P_LOG)]
linhas = {}
for nome, p in P_LOG_R.items():
    m = {k: float(v) for k, v in painel(confusoes(y, p.argmax(axis=1))[0]).items()}
    linhas[f"logística, entrada do {nome}"] = {"pior F1": m["pior F1"], **{f"F1 {c}": m[f"F1 {c}"] for c in CLASSES},
                                               "MCC": m["MCC"], "AUC-PR GS": average_precision_score(y == GS, p[:, GS]),
                                               "CIN → GS": cin_gs(p)}
tab_r = pd.concat([tab_r, pd.DataFrame(linhas).T])
tab_r.to_csv(RESULTS / "04_models_r1.csv")
display(tab_r.round(3))

comp = {}
for nome in R1:
    a = [PO[(nome, s)].argmax(axis=1) for s in SEMENTES]
    comp[f"{nome} − rede base"] = {m: dif_media(list(zip(a, [PO[(BASE_R, s)].argmax(axis=1) for s in SEMENTES])), m) for m in MET}
    comp[f"Dif({nome}): {nome} − logística com a mesma entrada"] = {m: dif_media([(q, P_LOG_R[nome].argmax(axis=1)) for q in a], m)
                                                                   for m in MET}
    comp[f"{nome} − logística atual"] = {m: dif_media([(q, P_LOG.argmax(axis=1)) for q in a], m) for m in MET}
    comp[f"logística com a entrada do {nome} − logística atual"] = {
        m: dif_media([(P_LOG_R[nome].argmax(axis=1), P_LOG.argmax(axis=1))], m) for m in MET}
print("Diferença (média das 5 sementes) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):")
pd.DataFrame(comp).T

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble,CIN → GS
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757,17.6
R1a,0.664,0.052,0.606,0.718,0.907,0.958,0.664,0.910,0.797,0.700,0.667,21.6
R1b,0.681,0.034,0.629,0.709,0.909,0.967,0.681,0.913,0.809,0.752,0.686,23.4
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN,18.0
"logística, entrada do R1a",0.750,NaN,NaN,NaN,0.922,0.946,0.750,0.951,0.837,0.771,NaN,18.0
"logística, entrada do R1b",0.733,NaN,NaN,NaN,0.922,0.946,0.733,0.967,0.834,0.738,NaN,17.0


Diferença (média das 5 sementes) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):


,pior F1,F1 GS,F1 EBV,MCC
R1a − rede base,-0.054 (-0.090 a -0.021),-0.054 (-0.090 a -0.021),-0.012 (-0.056 a +0.034),-0.037 (-0.060 a -0.016)
Dif(R1a): R1a − logística com a mesma entrada,-0.086 (-0.142 a -0.035),-0.086 (-0.142 a -0.035),-0.040 (-0.093 a +0.000),-0.040 (-0.078 a -0.003)
R1a − logística atual,-0.086 (-0.143 a -0.033),-0.086 (-0.143 a -0.033),-0.040 (-0.093 a +0.000),-0.045 (-0.083 a -0.008)
logística com a entrada do R1a − logística atual,+0.000 (-0.030 a +0.029),+0.000 (-0.030 a +0.029),+0.000 (+0.000 a +0.000),-0.005 (-0.028 a +0.016)
R1b − rede base,-0.036 (-0.072 a -0.000),-0.036 (-0.072 a -0.000),-0.009 (-0.047 a +0.030),-0.025 (-0.051 a +0.000)
Dif(R1b): R1b − logística com a mesma entrada,-0.052 (-0.109 a +0.009),-0.052 (-0.109 a +0.009),-0.054 (-0.102 a -0.014),-0.026 (-0.064 a +0.015)
R1b − logística atual,-0.069 (-0.125 a -0.015),-0.069 (-0.125 a -0.015),-0.038 (-0.077 a -0.007),-0.033 (-0.070 a +0.001)
logística com a entrada do R1b − logística atual,-0.017 (-0.064 a +0.020),-0.017 (-0.064 a +0.020),+0.016 (+0.000 a +0.048),-0.008 (-0.038 a +0.020)


### Leitura do R1 (execução de 2026-11-01)

Execução pelo nbconvert das 15:11 às 17:39, sem erro. O treino do R1a e do R1b levou 145 min, com pico de RAM de 3,1 GB no
`results/04_models.log`. O pré-registro está no diário (2026-11-01, "Pré-registro do R1a e do R1b"). Métrica principal: pior F1 no
argmax, média das 5 sementes, nas 336 previsões fora do fold (GS = 46, EBV = 29). Em todas as linhas, a classe que cai é o **GS**:
o pior F1 é o F1 do GS. A zona morta zerou 61,6% dos valores de CNV.

| modelo | pior F1 (média ± dp) | F1 CIN | F1 MSI | F1 GS | F1 EBV | MCC | AUC-PR GS | CIN → GS |
|---|---|---|---|---|---|---|---|---|
| rede base (`profundo + adamw_dro`) | 0,717 ± 0,015 | 0,926 | 0,962 | 0,717 | 0,922 | 0,834 | 0,778 | 17,6 |
| R1a (zona morta, 3 canais) | 0,664 ± 0,052 | 0,907 | 0,958 | 0,664 | 0,910 | 0,797 | 0,700 | 21,6 |
| R1b (zona morta + ganho/perda, 4 canais) | 0,681 ± 0,034 | 0,909 | 0,967 | 0,681 | 0,913 | 0,809 | 0,752 | 23,4 |
| logística atual (1 execução) | 0,750 | 0,924 | 0,956 | 0,750 | 0,951 | 0,842 | 0,788 | 18 |
| logística, entrada do R1a | 0,750 | 0,922 | 0,946 | 0,750 | 0,951 | 0,837 | 0,771 | 18 |
| logística, entrada do R1b | 0,733 | 0,922 | 0,946 | 0,733 | 0,967 | 0,834 | 0,738 | 17 |

Nas redes, o F1 por classe, o MCC e o CIN → GS são médias das 5 sementes. Pior F1 do ensemble (média das probabilidades das 5
sementes): rede base 0,757, R1a 0,667, R1b 0,686.

Diferenças (média das 5 sementes) com IC 95% pelo bootstrap pareado (os 2.000 sorteios do `05_eval`):

| comparação | pior F1 | MCC |
|---|---|---|
| R1a − rede base | −0,054 (−0,090 a −0,021) | −0,037 (−0,060 a −0,016) |
| R1b − rede base | −0,036 (−0,072 a −0,000) | −0,025 (−0,051 a +0,000) |
| Dif(R1a) = R1a − logística com a entrada do R1a | −0,086 (−0,142 a −0,035) | −0,040 (−0,078 a −0,003) |
| Dif(R1b) = R1b − logística com a entrada do R1b | −0,052 (−0,109 a +0,009) | −0,026 (−0,064 a +0,015) |
| logística com a entrada do R1a − logística atual | +0,000 (−0,030 a +0,029) | −0,005 (−0,028 a +0,016) |
| logística com a entrada do R1b − logística atual | −0,017 (−0,064 a +0,020) | −0,008 (−0,038 a +0,020) |

1. **A zona morta piorou a rede, o contrário da hipótese.** O R1a ficou 0,054 abaixo da rede base, com um IC que exclui o 0. O R1b
   ficou 0,036 abaixo, com o limite superior do IC em 0,000. Os CIN chamados de GS subiram de 17,6 para 21,6 (R1a) e 23,4 (R1b),
   em vez de cair como na regra do TCGA do P1 (de 18 para 8). O dp entre as sementes também subiu (0,052 e 0,034, contra 0,015), e
   o ensemble cai junto (0,667 e 0,686, contra 0,757).
2. **Os 4 canais recuperam parte da perda:** o R1b fica 0,017 acima do R1a na média, uma diferença sem comparação pareada calculada
   e do tamanho de um a dois GS. Mesmo assim, não volta ao nível da rede base.
3. **Na logística, a entrada nova não muda nada mensurável.** Com a entrada do R1a, o F1 do GS é o mesmo (0,750, com os mesmos 18
   CIN → GS), e só o MSI muda (0,956 → 0,946). Com a entrada do R1b, a diferença é −0,017 (IC −0,064 a +0,020). Dif(R1a) e Dif(R1b)
   são negativos: a rede continua abaixo da logística com a mesma entrada.
4. **A decisão não é tomada aqui.** O critério da C.3.2 (BBC-par > 0 contra a melhor tabular) roda no bloco F do `05_eval`. Pelo
   pré-registro, o R1-p só roda se a média do pior F1 do R1 passar da melhor tabular da família. Com 0,664 e 0,681, contra pelo
   menos 0,750, essa condição não foi atingida.
5. **Ruído:** com GS = 46, um GS a mais ou a menos acertado muda o F1 do GS em 0,01 a 0,02. A queda do R1a (−0,054) corresponde a
   uns 3 a 5 GS, e o IC exclui o 0. A do R1b fica no limite. A diferença entre o R1a e o R1b (0,017) está no ruído.

# Logísticas dos testes sem rede da PARTE C (P3a, P3b e P4)

Decisão do usuário (2026-11-01): todo modelo treina neste notebook. As logísticas do P3a e do P4 (bloco E do `05_eval`,
2026-10-31) e a do P3b saíram do `05_eval`, que agora lê as previsões gravadas em `results/oof_p/`. As variáveis também são calculadas lá, para a
regra do TCGA e a descrição.
- **P3a:** fração de genes com |log2| > 0,2 em cada cromossomo (1–22 e X) e a maior delas: 24 variáveis.
- **P3b:** o mesmo por braço. O braço p termina no fim da última banda p do cytoBand hg19 (`data/raw/`, baixado pelo
  `01_data`). Saem os braços p dos acrocêntricos (13, 14, 15, 21 e 22), que têm poucos ou nenhum gene (pré-registro no diário,
  2026-11-01).
- **P4:** 26 estatísticas globais por paciente (8 por canal sobre o z por canal do fold, mais a fração alterada e os genes
  mutados). P4a: só elas; P4b: junto das 53.454 colunas.
- Logística C = 1, peso balanceado, com as 53.454 colunas de hoje (menos no P4a) mais as variáveis, padronizadas com o treino
  do fold. As do P3a e do P4 têm de reproduzir o resultado registrado no bloco E: P3a sem nenhuma previsão diferente da
  logística atual (pior F1 0,750); P4a 0,672 (39/58); P4b 0,757 (78/103).

In [18]:
OOF_P = RESULTS / "oof_p"  # logísticas dos testes sem rede; apague um arquivo para treinar aquele fold de novo
OOF_P.mkdir(exist_ok=True)
info = pd.read_csv(RAIZ / "data" / "interim" / "genes.csv")
assert (info.gene.to_numpy() == layout.gene.to_numpy()).all()
CROM = info.cromossomo.astype(str).to_numpy()
CROMS = [str(c) for c in range(1, 23)] + ["X"]
alt, fin = np.abs(tensor[:, 1]) > 0.2, np.isfinite(tensor[:, 1])


def fracoes(grupo, nomes):
    # fração de genes com |log2| > 0,2 entre os medidos de cada grupo (cromossomo ou braço), por paciente
    return np.column_stack([alt[:, grupo == g].sum(axis=1) / np.maximum(fin[:, grupo == g].sum(axis=1), 1) for g in nomes])


FRAC_CROM = fracoes(CROM, CROMS)
cyto = pd.read_csv(RAIZ / "data" / "raw" / "cytoBand_hg19.txt.gz", sep="\t", header=None,
                   names=["chrom", "inicio", "fim", "banda", "coloracao"])
cyto["chrom"] = cyto.chrom.str.removeprefix("chr")
FIM_P = cyto[cyto.banda.str.startswith("p")].groupby("chrom").fim.max()  # o braço p termina no fim da última banda p
BRACO = np.array([c + ("p" if i < FIM_P[c] else "q") for c, i in zip(CROM, info.inicio)])
ACRO = ["13p", "14p", "15p", "21p", "22p"]  # braços p dos acrocêntricos: saem (pré-registro)
BRACOS = [c + b for c in CROMS for b in "pq" if (BRACO == c + b).any() and c + b not in ACRO]
FRAC_BRACO = fracoes(BRACO, BRACOS)
VARS_P = {"P3a": np.column_stack([FRAC_CROM, FRAC_CROM.max(axis=1)]), "P3b": np.column_stack([FRAC_BRACO, FRAC_BRACO.max(axis=1)])}
FRACAO, MUTADOS = alt.sum(axis=1) / fin.sum(axis=1), (tensor[:, 0] > 0).sum(axis=1)  # as variáveis da regra do TCGA (05_eval)


def estat_globais(k):
    # P4 (cópia do 05_eval): 8 estatísticas por canal sobre o z do fold k, como a Small CNN vê, mais a fração alterada e os
    # genes mutados: 26 por paciente
    z = padronizar(tensor, padronizacao[k])
    a = np.abs(z)
    return np.column_stack([z.mean(axis=2), z.std(axis=2), (a > 1).mean(axis=2), (a > 2).mean(axis=2), (a > 3).mean(axis=2),
                            *np.quantile(z, [0.01, 0.5, 0.99], axis=2), FRACAO, MUTADOS])


medidos = np.column_stack([fin[:, CROM == c].sum(axis=1) for c in CROMS])  # autoteste: a média pesada é a fração do genoma
assert np.allclose((FRAC_CROM * medidos).sum(axis=1) / medidos.sum(axis=1), FRACAO)
assert estat_globais(0).shape == (len(y), 26)
assert VARS_P["P3a"].shape[1] == 24 and all(np.isfinite(v).all() for v in VARS_P.values())
print(f"P3b: {len(BRACOS)} braços + a maior fração; genes nos braços que saem: {({a: int((BRACO == a).sum()) for a in ACRO})}")


def logistica_com(nome, extra, com_tensor=True):
    # logística C = 1 (cópia do 05_eval); extra(k) devolve as variáveis (336, m), padronizadas com o treino do fold k;
    # com_tensor junta antes delas as 53.454 colunas de hoje
    proba = np.empty((len(y), len(CLASSES)))
    for k in range(K):
        arq, te = OOF_P / f"{nome}_logistica_fold{k}.csv", fold == k
        if not arq.exists():
            log(f"{nome} · logística · fold {k}")
            v = extra(k)
            media, desvio = v[~te].mean(axis=0), v[~te].std(axis=0)
            partes = [(v - media) / np.where(desvio > 0, desvio, 1)]
            if com_tensor:
                partes.insert(0, padronizar(tensor, padronizacao[k]).reshape(len(y), -1))
            x = np.hstack(partes).astype(np.float32)
            p = treinar_logistica(x[~te], y[~te], x[te], SEED + k)
            pd.DataFrame(p, columns=PROBS).assign(amostra=coorte.amostra[te].to_numpy()).to_csv(arq, index=False)
            del x
        proba[te] = pd.read_csv(arq).set_index("amostra").loc[coorte.amostra[te], PROBS].to_numpy()
    return proba


t0 = time.time()
P_P = {nome: logistica_com(nome, lambda k, v=v: v) for nome, v in VARS_P.items()}
P_P["P4a"] = logistica_com("P4a", estat_globais, com_tensor=False)
P_P["P4b"] = logistica_com("P4b", estat_globais)
log(f"logísticas do P3a, P3b e P4 em {(time.time() - t0) / 60:.1f} min")
# autotestes: reproduzem o bloco E do 05_eval (2026-10-31)
assert np.array_equal(P_P["P3a"].argmax(axis=1), P_LOG.argmax(axis=1))  # P3a: nenhuma previsão diferente da logística atual
for n, ref in [("P4a", 39 / 58), ("P4b", 78 / 103)]:
    assert np.isclose(painel(confusoes(y, P_P[n].argmax(axis=1))[0])["pior F1"], ref), n
pd.DataFrame({n: {k: float(v) for k, v in painel(confusoes(y, p.argmax(axis=1))[0]).items()}
              for n, p in {"logística atual": P_LOG, **P_P}.items()}).T.round(3)

P3b: 41 braços + a maior fração; genes nos braços que saem: {'13p': 0, '14p': 0, '15p': 0, '21p': 0, '22p': 0}


22:46:54 [RAM 1396 MB, pico 2004 MB] logísticas do P3a, P3b e P4 em 0.0 min


,MCC,acurácia balanceada,F1 macro,pior F1,F1 CIN,F1 MSI,F1 GS,F1 EBV
logística atual,0.842,0.916,0.895,0.750,0.924,0.956,0.750,0.951
P3a,0.842,0.916,0.895,0.750,0.924,0.956,0.750,0.951
P3b,0.847,0.917,0.898,0.757,0.927,0.956,0.757,0.951
P4a,0.798,0.906,0.863,0.672,0.888,0.975,0.672,0.918
P4b,0.847,0.917,0.898,0.757,0.927,0.956,0.757,0.951


# Treinos que estavam no `05_eval` (blocos B, C e D)

Decisão do usuário (2026-11-01): todo treino fica neste notebook, e o `05_eval` só avalia, lendo o que está gravado. O código veio
do `05_eval` sem mudança de método. As previsões e os modelos já gravados são reaproveitados: nada é treinado de novo se os
arquivos existirem.
- **Bloco B:** as sementes 142, 242, 342 e 442 da Small CNN (Hilbert) e da Small CNN (vetor 1D), e as ablações por canal (só um
  canal; tirando um canal) na Small CNN (Hilbert) e na logística, semente 42 → `results/oof_b/`.
- **Bloco C:** os 5 modelos da Small CNN (Hilbert), semente 42, salvos para o Grad-CAM → `results/modelos_c/`.
- **Bloco D:** D2, controle de permutação gene → pixel (um sorteio por semente), e D3, z por gene → `results/oof_d/`. O sorteio
  `PERM` do D2 também serve aos controles de permutação dos testes R (R0-p, R9-p).

In [19]:
# bloco B do 05_eval: sementes extras e ablações por canal
OOF_B = RESULTS / "oof_b"
OOF_B.mkdir(exist_ok=True)
CONFIGS = {"so_mutacao": [0], "so_cnv": [1], "so_metilacao": [2],  # canais mantidos (0 mutação, 1 CNV, 2 metilação)
           "sem_mutacao": [1, 2], "sem_cnv": [0, 2], "sem_metilacao": [0, 1]}
EXPERIMENTOS = ([("cnn_hilbert", s, None) for s in SEMENTES[1:]]
                + [(m, SEED, c) for m in ["logistica", "cnn_hilbert"] for c in CONFIGS]
                + [("cnn_vetor", s, None) for s in SEMENTES[1:]])


def rodar_b(modelo, semente, config=None):
    # um arquivo por experimento e fold em results/oof_b/; reaproveita o que existe
    nome = f"{modelo}_s{semente}" + (f"_{config}" if config else "")
    formatar, treinar = MODELOS[modelo]
    for k in range(K):
        arq, te = OOF_B / f"{nome}_fold{k}.csv", fold == k
        if not arq.exists():
            log(f"bloco B: {nome} · fold {k}")
            z = padronizar(tensor, padronizacao[k])
            if config:
                z[:, [c for c in range(3) if c not in CONFIGS[config]]] = 0  # canal retirado = média do treino
            x = formatar(z)
            p = treinar(x[~te], y[~te], x[te], semente + k)
            pd.DataFrame(p, columns=PROBS).assign(amostra=coorte.amostra[te].to_numpy()).to_csv(arq, index=False)
            del z, x


for e in EXPERIMENTOS:
    rodar_b(*e)

# bloco C do 05_eval: os 5 modelos da Small CNN (Hilbert), semente 42, salvos para o Grad-CAM
MODELOS_C = RESULTS / "modelos_c"
MODELOS_C.mkdir(exist_ok=True)
for k in range(K):
    arq = MODELOS_C / f"cnn_hilbert_s{SEED}_fold{k}.keras"
    if not arq.exists():
        log(f"bloco C: cnn_hilbert_s{SEED} · fold {k}, salvando o modelo")
        x = MODELOS["cnn_hilbert"][0](padronizar(tensor, padronizacao[k]))
        treinar_cnn(x[fold != k], y[fold != k], x[fold == k], SEED + k, guardar=arq)
        del x
assert len(list(OOF_B.glob("*.csv"))) == 5 * len(EXPERIMENTOS) and len(list(MODELOS_C.glob("*.keras"))) == K
print(f"bloco B: {5 * len(EXPERIMENTOS)} arquivos em results/oof_b/ · bloco C: {K} modelos em results/modelos_c/")

bloco B: 100 arquivos em results/oof_b/ · bloco C: 5 modelos em results/modelos_c/


In [20]:
import warnings

# bloco D do 05_eval: D2 (permutação gene → pixel) e D3 (z por gene)
OOF_D = RESULTS / "oof_d"
OOF_D.mkdir(exist_ok=True)
N = tensor.shape[2]
PERM = {s: np.random.default_rng(1000 + s).permutation(N) for s in SEMENTES}  # um sorteio por semente, o mesmo nos 5 folds,
assert all(np.array_equal(np.sort(p), np.arange(N)) and (p != np.arange(N)).mean() > 0.99 for p in PERM.values())  # canais e pacientes
lin, col = layout.linha.to_numpy(), layout.coluna.to_numpy()
z0 = padronizar(tensor[:2], padronizacao[0])
g = layout.gene.tolist().index("MLH1")
assert np.array_equal(para_hilbert(z0[:, :, PERM[SEED]])[:, :, lin[g], col[g]], z0[:, :, PERM[SEED][g]])  # o pixel do MLH1 guarda outro gene


def stats_gene(k):
    # média e desvio de cada gene no treino do fold (mutação em log1p antes, como no z por canal)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # metilação do X: gene todo NaN
        xt = transformar(tensor[fold != k])
        return np.nanmean(xt, axis=0), np.nanstd(xt, axis=0)  # (3, 17.818) cada


STATS_GENE = {k: stats_gene(k) for k in range(K)}
VARIANTES_D = {"zgene_cnv_metil": (1, 2), "zgene": (0, 1, 2)}  # canais com z por gene; os outros seguem com z por canal


def padronizar_gene(x, k, canais):
    z, xt = padronizar(x, padronizacao[k]), transformar(x)
    media, desvio = STATS_GENE[k]
    for c in canais:
        d = np.where(desvio[c] > 0, desvio[c], np.inf)  # gene constante no treino (nunca mutado; metilação do X): z = 0
        z[:, c] = np.nan_to_num((xt[:, c] - media[c]) / d, nan=0.0)
    return z


# autoteste: no treino de cada fold, cada gene sem NaN e com variação fica com média 0 e desvio 1
for k in range(K):
    zt, desvio = padronizar_gene(tensor[fold != k], k, (0, 1, 2)), STATS_GENE[k][1]
    ok = (desvio > 0) & ~np.isnan(tensor[fold != k]).any(axis=0)
    assert np.allclose(zt.mean(axis=0)[ok], 0, atol=1e-3) and np.allclose(zt.std(axis=0)[ok], 1, atol=1e-3), k
del zt, z0

EXP_D = ([(f"cnn_hilbert_perm_s{s}", "cnn_hilbert", s, lambda k, s=s: padronizar(tensor, padronizacao[k])[:, :, PERM[s]])
          for s in SEMENTES]
         + [(f"{m}_{v}_s{s}", m, s, lambda k, c=c: padronizar_gene(tensor, k, c))
            for v, c in VARIANTES_D.items() for m, sem in [("logistica", [SEED]), ("cnn_hilbert", SEMENTES)] for s in sem])


def rodar_d(nome, modelo, semente, preparar):
    # como a rodar_b(); preparar(k) devolve o tensor padronizado (336, 3, 17.818) do fold k
    formatar, treinar = MODELOS[modelo]
    for k in range(K):
        arq, te = OOF_D / f"{nome}_fold{k}.csv", fold == k
        if not arq.exists():
            log(f"bloco D: {nome} · fold {k}")
            x = formatar(preparar(k))
            p = treinar(x[~te], y[~te], x[te], semente + k)
            pd.DataFrame(p, columns=PROBS).assign(amostra=coorte.amostra[te].to_numpy()).to_csv(arq, index=False)
            del x


for nome, modelo, semente, preparar in EXP_D:
    rodar_d(nome, modelo, semente, preparar)
assert len(EXP_D) == 17 and len(list(OOF_D.glob("*.csv"))) == 5 * len(EXP_D)
print(f"bloco D: {5 * len(EXP_D)} arquivos em results/oof_d/")

bloco D: 85 arquivos em results/oof_d/


# R9 — saída achatada (Flatten), com os controles R9-p e R0-p (2026-11-01)

PARTE C de `docs/relatorio_imagem.txt` (R9); pré-registro no diário (2026-11-01, "Pré-registro do R9, do R9-p e do R0-p").
Rótulo PRIORIDADE (`.claude/rules/regras_de_testes.md`); roda antes do R10 (decisão do usuário, 2026-11-01).

- **R9:** a rede base (`profundo + adamw_dro`) com Flatten depois do último MaxPool (8 × 8 × 64 = 4.096 números) no lugar do
  GlobalAveragePooling; o mesmo Dropout 0,3 e a mesma Dense(4). Com a média global, a rede só sabe quanto de cada padrão há no
  genoma; com o Flatten, cada uma das 64 posições do mapa final tem peso próprio. Uma posição junta 256 genes, em ~11 trechos do
  genoma (contado pelo cebolinha): é uma vizinhança aproximada, não um trecho contínuo.
- **R9-p:** a mesma rede com os genes sorteados nos pixels (o `PERM` do D2, um sorteio por semente). Perm(R9) = R9 − R9-p mede se
  a ordem do genoma importa quando a rede usa a posição.
- **R0-p:** a rede base (com a média global) com o mesmo sorteio. A C.3.4 exige esse controle porque a rede base não é o
  `profundo` (o D2 foi treinado com o `profundo`).
- A entrada é a atual, então a logística de comparação é a atual (0,750). Risco declarado: 16.388 pesos na saída para ~269
  pacientes de treino por fold.

In [21]:
CFG_R9 = BASE | {"achatar": True}
assert rede((*FORMA, 3)).count_params() == 61476 and rede((*FORMA, 3), achatar=True).count_params() == 77604
x = FORMATOS["hilbert"](padronizar(tensor[sub], padronizacao[0]))  # autoteste: 1 época em 40 pacientes
p = treinar_variante(x, y[sub], x[:3], SEED, **CFG_R9, epocas=1)
assert p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4)
del x, p


def permutado(s):
    # entrada atual com os genes sorteados nos pixels: o sorteio da semente s do D2, o mesmo nos 5 folds
    return lambda k: padronizar(tensor, padronizacao[k])[:, :, PERM[s]]


TESTES_R9 = {"R9": (CFG_R9, False), "R9-p": (CFG_R9, True), "R0-p": (BASE, True)}  # nome → (rede, genes sorteados?)
log("=== R9, R9-p e R0-p ===")
t0 = time.time()
for nome, (cfg, sorteia) in TESTES_R9.items():
    for s in SEMENTES:
        log(f"{nome}: semente {s}")
        PO[(nome, s)] = rodar_o(nome, s, cfg, preparar=permutado(s) if sorteia else None, pasta=OOF_R)
log(f"R9, R9-p e R0-p terminados em {(time.time() - t0) / 60:.0f} min")

22:47:33 [RAM 1426 MB, pico 2004 MB]   1 épocas, melhor F1 macro na validação interna 0.071


22:47:35 [RAM 1427 MB, pico 2004 MB] === R9, R9-p e R0-p ===
22:47:35 [RAM 1427 MB, pico 2004 MB] R9: semente 42


22:47:35 [RAM 1427 MB, pico 2004 MB] R9: semente 142


22:47:35 [RAM 1427 MB, pico 2004 MB] R9: semente 242


22:47:36 [RAM 1427 MB, pico 2004 MB] R9: semente 342


22:47:36 [RAM 1427 MB, pico 2004 MB] R9: semente 442


... (5 outputs left out)

22:47:38 [RAM 1427 MB, pico 2004 MB] R0-p: semente 142


22:47:38 [RAM 1427 MB, pico 2004 MB] R0-p: semente 242


22:47:38 [RAM 1427 MB, pico 2004 MB] R0-p: semente 342


22:47:38 [RAM 1427 MB, pico 2004 MB] R0-p: semente 442
22:47:39 [RAM 1427 MB, pico 2004 MB] R9, R9-p e R0-p terminados em 0 min


In [22]:
V9 = [BASE_R, *TESTES_R9]
tab_9 = resumo(V9)
tab_9["CIN → GS"] = [np.mean([cin_gs(PO[(v, s)]) for s in SEMENTES]) for v in V9] + [cin_gs(P_LOG)]
tab_9.to_csv(RESULTS / "04_models_r9.csv")
display(tab_9.round(3))


def pares(a, b):
    return list(zip([PO[(a, s)].argmax(axis=1) for s in SEMENTES], [PO[(b, s)].argmax(axis=1) for s in SEMENTES]))


comp9 = {"Perm(R9) = R9 − R9-p": ("R9", "R9-p"), "R9 − rede base": ("R9", BASE_R), "R9-p − R0-p": ("R9-p", "R0-p"),
         "R0-p − rede base": ("R0-p", BASE_R)}
print("Diferença pareada por semente (média das 5) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):")
pd.DataFrame({n: {m: dif_media(pares(a, b), m) for m in MET} for n, (a, b) in comp9.items()}).T

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble,CIN → GS
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757,17.6
R9,0.699,0.037,0.636,0.727,0.917,0.949,0.699,0.927,0.815,0.767,0.720,18.4
R9-p,0.692,0.016,0.673,0.709,0.907,0.956,0.692,0.917,0.807,0.765,0.692,22.8
R0-p,0.697,0.037,0.642,0.731,0.908,0.949,0.697,0.882,0.802,0.765,0.729,22.8
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN,18.0


Diferença pareada por semente (média das 5) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):


,pior F1,F1 GS,F1 EBV,MCC
Perm(R9) = R9 − R9-p,+0.008 (-0.028 a +0.045),+0.008 (-0.028 a +0.045),+0.010 (-0.020 a +0.041),+0.008 (-0.014 a +0.032)
R9 − rede base,-0.018 (-0.060 a +0.024),-0.018 (-0.060 a +0.024),+0.005 (-0.027 a +0.034),-0.019 (-0.043 a +0.007)
R9-p − R0-p,-0.006 (-0.036 a +0.026),-0.006 (-0.036 a +0.026),+0.036 (+0.005 a +0.071),+0.005 (-0.013 a +0.024)
R0-p − rede base,-0.020 (-0.050 a +0.009),-0.020 (-0.050 a +0.010),-0.041 (-0.075 a -0.011),-0.032 (-0.053 a -0.011)


### Leitura do R9, do R9-p e do R0-p (execução de 2026-11-01)

Execução pelo nbconvert das 19:37 às 22:27. O treino levou 167 min (R9 19:40–20:25, R9-p 20:25–21:18, R0-p 21:18–22:27), dentro
dos ~3 h 15 do pré-registro, com pico de RAM de 3,5 GB no `results/04_models.log`. Sem erro, com 22 células de código numeradas
de 1 a 22 (conferência da sessão principal, no diário: "Resultado do R9, do R9-p e do R0-p e do bloco G"). O pré-registro está no
diário (2026-11-01, "Pré-registro do R9, do R9-p e do R0-p"). Métrica principal: pior F1 no argmax, média das 5 sementes, nas 336
previsões fora do fold (GS = 46, EBV = 29). Em todas as linhas, a classe que cai é o **GS**: o pior F1 é o F1 do GS.

| modelo | pior F1 (média ± dp) | mín.–máx. | F1 CIN | F1 MSI | F1 GS | F1 EBV | MCC | AUC-PR GS | CIN → GS |
|---|---|---|---|---|---|---|---|---|---|
| rede base (`profundo + adamw_dro`, média global) | 0,717 ± 0,015 | 0,692–0,733 | 0,926 | 0,962 | 0,717 | 0,922 | 0,834 | 0,778 | 17,6 |
| R9 (Flatten, genes na ordem do genoma) | 0,699 ± 0,037 | 0,636–0,727 | 0,917 | 0,949 | 0,699 | 0,927 | 0,815 | 0,767 | 18,4 |
| R9-p (Flatten, genes sorteados) | 0,692 ± 0,016 | 0,673–0,709 | 0,907 | 0,956 | 0,692 | 0,917 | 0,807 | 0,765 | 22,8 |
| R0-p (média global, genes sorteados) | 0,697 ± 0,037 | 0,642–0,731 | 0,908 | 0,949 | 0,697 | 0,882 | 0,802 | 0,765 | 22,8 |
| logística (1 execução) | 0,750 | — | 0,924 | 0,956 | 0,750 | 0,951 | 0,842 | 0,788 | 18 |

Nas redes, o F1 por classe, o MCC e o CIN → GS são médias das 5 sementes. Pior F1 do ensemble (média das probabilidades das 5
sementes): rede base 0,757, R9 0,720, R9-p 0,692, R0-p 0,729.

Diferenças (média das 5 sementes) com IC 95% pelo bootstrap pareado (os 2.000 sorteios do `05_eval`). O F1 do GS é igual ao pior F1
em todas as linhas.

| comparação | pior F1 | F1 EBV | MCC |
|---|---|---|---|
| Perm(R9) = R9 − R9-p | +0,008 (−0,028 a +0,045) | +0,010 (−0,020 a +0,041) | +0,008 (−0,014 a +0,032) |
| R9 − rede base | −0,018 (−0,060 a +0,024) | +0,005 (−0,027 a +0,034) | −0,019 (−0,043 a +0,007) |
| R9-p − R0-p | −0,006 (−0,036 a +0,026) | +0,036 (+0,005 a +0,071) | +0,005 (−0,013 a +0,024) |
| R0-p − rede base | −0,020 (−0,050 a +0,009) | −0,041 (−0,075 a −0,011) | −0,032 (−0,053 a −0,011) |

1. **Desfecho pelo pré-registro: REFUTA, pelas duas condições.** O IC de Perm(R9) contém o 0 (+0,008, −0,028 a +0,045), e o R9
   ficou abaixo da rede base (−0,018). Com a posição na saída, a rede não separa blocos que são vizinhanças do genoma de blocos com
   genes sorteados. Pela PARTE C (R9), o nível 3 fecha para esta saída.
2. **A melhor variante do teste é o próprio R9 (0,699 ± 0,037), e ela não passa da rede base:**
   - contra a rede base: −0,018 (IC −0,060 a +0,024), com o 0 no IC;
   - contra a logística (0,750): −0,051 na média. A diferença pareada R9 − logística não estava no pré-registro e não foi calculada.
     A comparação com a tabular é a BBC-par do bloco G do `05_eval`: −0,039 (IC −0,120 a +0,036), também com o 0 no IC;
   - o dp entre as sementes passou de 0,015 para 0,037, a pior semente caiu para 0,636 e o ensemble caiu de 0,757 para 0,720.
3. **Sobreajuste da saída (a leitura pré-registrada):** o R9 (0,699) e o R9-p (0,692) ficaram os dois abaixo da rede base (0,717),
   que é a condição do pré-registro para essa leitura (16.388 pesos na Dense para ~269 pacientes de treino por fold). Ressalva: o
   R9-p fica abaixo da base também por causa do sorteio dos genes, porque o R0-p, sem o Flatten, cai o mesmo tanto (−0,020). As duas
   comparações que isolam o Flatten (R9 − rede base, −0,018; R9-p − R0-p, −0,006) têm o 0 no IC. O que aponta para o sobreajuste é o
   dp maior e a queda do ensemble, sem teste formal.
4. **R0-p, o controle da rede base:** no pior F1, sortear os genes não muda a rede base de forma mensurável (−0,020, IC −0,050 a
   +0,009), como no D2 com o `profundo` (+0,003, IC −0,025 a +0,032). Já o MCC (−0,032, IC −0,053 a −0,011) e o F1 do EBV (−0,041,
   IC −0,075 a −0,011) caem com o IC sem o 0, e os CIN → GS sobem de 17,6 para 22,8. O F1 do EBV do R0-p (0,882) é o mais baixo da
   tabela, e por isso o R9-p − R0-p no EBV também sai do 0 (+0,036). São métricas secundárias, em 4 comparações × 4 métricas sem
   correção para comparações múltiplas, e não entram na decisão do pré-registro.
5. **Ruído:** com GS = 46, um GS a mais ou a menos acertado muda o F1 do GS em 0,01 a 0,02. As diferenças no pior F1 deste teste
   (de 0,006 a 0,020) ficam nessa faixa, e nenhum IC do pior F1 exclui o 0. Diferenças de 0,02 a 0,05 costumam estar no ruído.
6. **A decisão sobre a imagem é do bloco G do `05_eval`** (BBC-par, critério da C.3.2).

# R2 — CNV corrigida pela pureza (2026-11-03)

PARTE C de `docs/relatorio_imagem.txt` (R2); pré-registro no diário (2026-11-03, "Pré-registro do R2"). Rótulo DEFINIDO, o 1º da
fila (`.claude/rules/regras_de_testes.md`); liberado pelo P2 (os CIN que a Small CNN chamou de GS têm pureza mediana 0,27, contra
0,56 nos outros CIN).

- **Hipótese:** parte dos CIN → GS são CIN de pureza baixa: a mistura com tecido normal encolhe os ganhos e as perdas de braço,
  que são o que separa o CIN do GS.
- **Entrada:** na CNV bruta, R = 2^log2 e R' = (R − (1 − p)) / p, com p = pureza ABSOLUTE truncada em [0,2; 1] e R' com piso de
  0,125 (log2 = −3); depois log2(R'). Os 8 pacientes sem pureza (6 GS, 1 CIN e 1 MSI) ficam com p = 1, sem correção (decisão do
  usuário, 2026-11-03). Mutação em `log1p` e metilação como hoje; z por canal com o treino de cada fold; `NaN` → 0 depois.
- A logística com a mesma entrada roda antes da rede. Dif(R2) = pior F1 da rede − pior F1 dessa logística.

In [23]:
PUREZA = (pd.read_csv(RAIZ / "data" / "raw" / "TCGA_mastercalls.abs_tables_JSedit.fixed.txt", sep="\t")
          .set_index("array").purity.reindex(coorte.amostra).to_numpy())        # 8 pacientes sem pureza
P_R2 = np.clip(np.nan_to_num(PUREZA, nan=1.0), 0.2, 1.0)[:, None]              # sem pureza → 1: sem correção


def corrigir(log2, p):
    # CNV corrigida pela pureza p (PARTE C, R2): R = 2^log2, R' = (R − (1 − p)) / p, piso de 0,125 (log2 = −3).
    # Escrita como 1 + (R − 1) / p, que é a mesma conta: assim log2 = 0 continua 0 exato (na outra ordem, 1 − (1 − p)
    # não dá p exato, e 2.661 dos 8.088 zeros da CNV viravam ±1e-16). Em float64: em float32, log2(2^x) ≠ x perto de 0
    log2 = np.asarray(log2, np.float64)
    return np.log2(np.maximum(1 + (2.0 ** log2 - 1) / p, 0.125))


def entrada_r2(x):
    # valores antes do z: mutação em log1p; CNV corrigida pela pureza de cada paciente; NaN continua NaN
    x = transformar(x)
    x[:, 1] = corrigir(x[:, 1], P_R2)
    return x


def preparar_r2(k):
    # z por canal com a média e o desvio do treino do fold k (ignorando NaN); NaN → 0 depois, como em preparar_r1()
    x = entrada_r2(tensor)
    tr = x[fold != k]
    return np.nan_to_num((x - np.nanmean(tr, axis=(0, 2), keepdims=True)) / np.nanstd(tr, axis=(0, 2), keepdims=True), nan=0.0)


# autoteste, antes da hora de treino. Contas à mão com p = 0,5: log2 1 → R' = (2 − 0,5) / 0,5 = 3; log2 0 → 1;
# log2 −1 → R' = 0 → piso; com p = 1, só o piso muda o valor
assert np.allclose(corrigir(np.array([1.0, 0.0, -1.0, -5.0, np.nan]), 0.5), [np.log2(3), 0, -3, -3, np.nan], equal_nan=True)
assert np.allclose(corrigir(np.array([0.7, -0.4, -4.0]), 1.0), [0.7, -0.4, -3])
sem = np.isnan(PUREZA)
x0, x2 = transformar(tensor), entrada_r2(tensor)
assert sem.sum() == 8 and (np.isnan(x2) == np.isnan(x0)).all()                  # 8 sem pureza; nenhum NaN criado ou apagado
assert np.allclose(x2[:, [0, 2]], x0[:, [0, 2]], equal_nan=True)                # mutação e metilação iguais às de hoje
assert np.allclose(x2[sem, 1], x0[sem, 1], atol=1e-6, equal_nan=True)            # sem pureza: CNV como hoje
ok = ~np.isnan(x0[:, 1])
assert (np.sign(x2[:, 1][ok]) == np.sign(x0[:, 1][ok])).all()                    # não troca ganho por perda; 0 continua 0
print(f"CNV no piso de −3: {(x2[:, 1] == -3).sum()} valores ({(x0[:, 1] < -3).sum()} já eram < −3); desvio "
      f"{np.nanstd(x0[:, 1]):.3f} → {np.nanstd(x2[:, 1]):.3f}; pureza < 0,2 (vira 0,2): {int((PUREZA < 0.2).sum())} pacientes")
del x0, x2, ok
for k in range(K):
    z = preparar_r2(k)
    assert np.allclose(z[:, [0, 2]], padronizar(tensor, padronizacao[k])[:, [0, 2]], atol=1e-4), k
    assert np.allclose(z[fold != k].mean(axis=(0, 2)), 0, atol=1e-3), k           # média 0 no treino do fold
x = FORMATOS["hilbert"](z[sub])                                                    # 1 época em 40 pacientes
p = treinar_variante(x, y[sub], x[:3], SEED, **BASE, epocas=1)
assert p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4)
del z, x, p

CNV no piso de −3: 3994 valores (78 já eram < −3); desvio 0.212 → 0.409; pureza < 0,2 (vira 0,2): 20 pacientes


22:48:33 [RAM 1624 MB, pico 2004 MB]   1 épocas, melhor F1 macro na validação interna 0.071


In [24]:
log("=== R2 ===")
t0 = time.time()
P_LOG_R2 = np.empty((len(y), len(CLASSES)))  # logística com a mesma entrada, antes da rede (C.2)
for k in range(K):
    arq, te = OOF_R / f"R2_logistica_fold{k}.csv", fold == k
    if not arq.exists():
        log(f"R2 · logística · fold {k}")
        x = preparar_r2(k).reshape(len(y), -1)
        p = treinar_logistica(x[~te], y[~te], x[te], SEED + k)
        pd.DataFrame(p, columns=PROBS).assign(amostra=coorte.amostra[te].to_numpy()).to_csv(arq, index=False)
        del x
    P_LOG_R2[te] = pd.read_csv(arq).set_index("amostra").loc[coorte.amostra[te], PROBS].to_numpy()
for s in SEMENTES:
    log(f"R2: semente {s}")
    PO[("R2", s)] = rodar_o("R2", s, BASE, preparar=preparar_r2, pasta=OOF_R)
log(f"R2 terminado em {(time.time() - t0) / 60:.0f} min")

22:48:34 [RAM 1557 MB, pico 2004 MB] === R2 ===


22:48:35 [RAM 1557 MB, pico 2004 MB] R2: semente 42


22:48:35 [RAM 1557 MB, pico 2004 MB] R2: semente 142


22:48:35 [RAM 1557 MB, pico 2004 MB] R2: semente 242


22:48:35 [RAM 1557 MB, pico 2004 MB] R2: semente 342


22:48:35 [RAM 1557 MB, pico 2004 MB] R2: semente 442


22:48:36 [RAM 1557 MB, pico 2004 MB] R2 terminado em 0 min


In [25]:
def gs_cin(p):
    return int(confusoes(y, p.argmax(axis=1))[0][GS, CIN])  # GS previstos como CIN


V2 = [BASE_R, "R2"]
tab_2 = resumo(V2)
tab_2["CIN → GS"] = [np.mean([cin_gs(PO[(v, s)]) for s in SEMENTES]) for v in V2] + [cin_gs(P_LOG)]
tab_2["GS → CIN"] = [np.mean([gs_cin(PO[(v, s)]) for s in SEMENTES]) for v in V2] + [gs_cin(P_LOG)]
m = {k: float(v) for k, v in painel(confusoes(y, P_LOG_R2.argmax(axis=1))[0]).items()}
tab_2.loc["logística, entrada do R2"] = pd.Series({"pior F1": m["pior F1"], **{f"F1 {c}": m[f"F1 {c}"] for c in CLASSES},
                                                   "MCC": m["MCC"], "AUC-PR GS": average_precision_score(y == GS, P_LOG_R2[:, GS]),
                                                   "CIN → GS": cin_gs(P_LOG_R2), "GS → CIN": gs_cin(P_LOG_R2)})
tab_2.to_csv(RESULTS / "04_models_r2.csv")
display(tab_2.round(3))

a = [PO[("R2", s)].argmax(axis=1) for s in SEMENTES]
log_r2, log_atual = P_LOG_R2.argmax(axis=1), P_LOG.argmax(axis=1)
comp2 = {"R2 − rede base": list(zip(a, [PO[(BASE_R, s)].argmax(axis=1) for s in SEMENTES])),
         "Dif(R2): R2 − logística com a mesma entrada": [(q, log_r2) for q in a],
         "R2 − logística atual": [(q, log_atual) for q in a],
         "logística com a entrada do R2 − logística atual": [(log_r2, log_atual)]}
print("Diferença (média das 5 sementes) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):")
pd.DataFrame({n: {m: dif_media(p, m) for m in MET} for n, p in comp2.items()}).T

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble,CIN → GS,GS → CIN
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757,17.6,8.2
R2,0.766,0.023,0.731,0.796,0.934,0.952,0.766,0.902,0.849,0.850,0.827,16.2,5.0
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN,18.0,6.0
"logística, entrada do R2",0.845,NaN,NaN,NaN,0.950,0.965,0.845,0.951,0.894,0.849,NaN,10.0,4.0


Diferença (média das 5 sementes) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):


,pior F1,F1 GS,F1 EBV,MCC
R2 − rede base,+0.048 (-0.004 a +0.102),+0.048 (-0.003 a +0.102),-0.020 (-0.055 a +0.014),+0.016 (-0.014 a +0.045)
Dif(R2): R2 − logística com a mesma entrada,-0.080 (-0.143 a -0.019),-0.080 (-0.144 a -0.019),-0.049 (-0.097 a -0.001),-0.045 (-0.086 a -0.005)
R2 − logística atual,+0.016 (-0.055 a +0.085),+0.016 (-0.055 a +0.085),-0.049 (-0.097 a -0.001),+0.007 (-0.037 a +0.050)
logística com a entrada do R2 − logística atual,+0.095 (+0.038 a +0.159),+0.095 (+0.038 a +0.160),+0.000 (+0.000 a +0.000),+0.052 (+0.020 a +0.087)


# R3 — metilação em valor M, centrada por gene (2026-11-03)

PARTE C de `docs/relatorio_imagem.txt` (R3); pré-registro no diário (2026-11-03, "Pré-registro do R3"). Rótulo DEFINIDO, o 2º da
fila (`.claude/rules/regras_de_testes.md`).

- **Hipótese:** centrado pela mediana do gene, o pixel de metilação passa a dizer "acima do normal para este gene". A
  hipermetilação do MLH1 no MSI e o CIMP do EBV se destacam sem ampliar o ruído, porque a escala é uma só para o canal inteiro.
- **Entrada:** mutação e CNV como hoje. Metilação: beta cortado em [0,001; 0,999], M = log2(beta / (1 − beta)), menos a mediana do
  gene no treino do fold, dividido pelo desvio dos resíduos do canal no treino do fold (um número só); `NaN` → 0.
- A logística com a mesma entrada roda antes da rede. As funções `rodar_valor()` e `tabela_valor()` servem também ao R4.

In [26]:
import warnings


def metil_m(beta):
    # valor M da metilação (PARTE C, R3): beta cortado em [0,001; 0,999], M = log2(beta / (1 − beta)); NaN continua NaN
    b = np.clip(np.asarray(beta, np.float64), 0.001, 0.999)
    return np.log2(b / (1 - b))


M_R3 = metil_m(tensor[:, 2])  # (336, 17818); os 825 genes do X não têm metilação, como hoje


def preparar_r3(k):
    # mutação e CNV como hoje; metilação em M, menos a mediana do gene no treino do fold k, dividida pelo desvio dos
    # resíduos do canal no treino (um número só); NaN → 0
    z, tr = padronizar(tensor, padronizacao[k]), fold != k
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # gene sem nenhum valor no treino: mediana NaN, vira 0 no fim
        res = M_R3 - np.nanmedian(M_R3[tr], axis=0)
    z[:, 2] = np.nan_to_num(res / np.nanstd(res[tr]), nan=0.0)
    return z


# autoteste, antes da hora de treino. Contas à mão: beta 0,5 → M 0; 0,8 → log2(0,8 / 0,2) = 2; 0 e 1 → o corte, ∓log2(999)
assert np.allclose(metil_m(np.array([0.5, 0.8, 0.0, 1.0, np.nan])), [0, 2, -np.log2(999), np.log2(999), np.nan], equal_nan=True)
tem = ~np.isnan(tensor[:, 2])
assert (np.isnan(M_R3) == ~tem).all() and (~tem).all(axis=0).sum() == 825
genes = tem.any(axis=0)
for k in range(K):
    z, tr = preparar_r3(k), fold != k
    assert np.allclose(z[:, :2], padronizar(tensor, padronizacao[k])[:, :2])        # mutação e CNV iguais às de hoje
    zm = np.where(tem, z[:, 2], np.nan)
    assert np.isclose(np.nanstd(zm[tr]), 1, atol=1e-3), k                            # desvio 1 no treino do fold
    assert np.allclose(np.nanmedian(zm[tr][:, genes], axis=0), 0, atol=1e-6), k       # mediana 0 em cada gene no treino
    assert (z[:, 2][~tem] == 0).all(), k                                               # NaN → 0
print(f"metilação: {(~tem).mean():.2%} NaN, {(~genes).sum()} genes sem nenhum valor; beta de {np.nanmin(tensor[:, 2]):.4f} "
      f"a {np.nanmax(tensor[:, 2]):.4f} (o corte em [0,001; 0,999] não muda nenhum valor)")
x = FORMATOS["hilbert"](z[sub])                                                        # 1 época em 40 pacientes
p = treinar_variante(x, y[sub], x[:3], SEED, **BASE, epocas=1)
assert p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4)
del z, zm, x, p, tem, genes

metilação: 4.65% NaN, 825 genes sem nenhum valor; beta de 0.0063 a 0.9917 (o corte em [0,001; 0,999] não muda nenhum valor)


22:49:37 [RAM 1640 MB, pico 2004 MB]   1 épocas, melhor F1 macro na validação interna 0.071


In [27]:
def rodar_valor(nome, preparar):
    # teste de valor do pixel (R3 e R4, como o R2): a logística com a mesma entrada, antes da rede (C.2), e a rede base nas
    # 5 sementes; previsões em results/oof_r/
    log(f"=== {nome} ===")
    t0, p_log = time.time(), np.empty((len(y), len(CLASSES)))
    for k in range(K):
        arq, te = OOF_R / f"{nome}_logistica_fold{k}.csv", fold == k
        if not arq.exists():
            log(f"{nome} · logística · fold {k}")
            x = preparar(k).reshape(len(y), -1)
            p = treinar_logistica(x[~te], y[~te], x[te], SEED + k)
            pd.DataFrame(p, columns=PROBS).assign(amostra=coorte.amostra[te].to_numpy()).to_csv(arq, index=False)
            del x
        p_log[te] = pd.read_csv(arq).set_index("amostra").loc[coorte.amostra[te], PROBS].to_numpy()
    for s in SEMENTES:
        log(f"{nome}: semente {s}")
        PO[(nome, s)] = rodar_o(nome, s, BASE, preparar=preparar, pasta=OOF_R)
    log(f"{nome} terminado em {(time.time() - t0) / 60:.0f} min")
    return p_log


P_LOG_R3 = rodar_valor("R3", preparar_r3)

22:49:38 [RAM 1576 MB, pico 2004 MB] === R3 ===


22:49:38 [RAM 1576 MB, pico 2004 MB] R3: semente 42
22:49:38 [RAM 1576 MB, pico 2004 MB] R3: semente 142


22:49:39 [RAM 1576 MB, pico 2004 MB] R3: semente 242
22:49:39 [RAM 1576 MB, pico 2004 MB] R3: semente 342


22:49:39 [RAM 1576 MB, pico 2004 MB] R3: semente 442
22:49:39 [RAM 1576 MB, pico 2004 MB] R3 terminado em 0 min


In [28]:
def tabela_valor(nome, p_log):
    # painel e diferenças com IC, como no R2; grava results/04_models_<nome>.csv
    v = [BASE_R, nome]
    tab = resumo(v)
    tab["CIN → GS"] = [np.mean([cin_gs(PO[(u, s)]) for s in SEMENTES]) for u in v] + [cin_gs(P_LOG)]
    tab["GS → CIN"] = [np.mean([gs_cin(PO[(u, s)]) for s in SEMENTES]) for u in v] + [gs_cin(P_LOG)]
    mt = {k: float(x) for k, x in painel(confusoes(y, p_log.argmax(axis=1))[0]).items()}
    tab.loc[f"logística, entrada do {nome}"] = pd.Series({"pior F1": mt["pior F1"], **{f"F1 {c}": mt[f"F1 {c}"] for c in CLASSES},
                                                         "MCC": mt["MCC"], "AUC-PR GS": average_precision_score(y == GS, p_log[:, GS]),
                                                         "CIN → GS": cin_gs(p_log), "GS → CIN": gs_cin(p_log)})
    tab.to_csv(RESULTS / f"04_models_{nome.lower()}.csv")
    a = [PO[(nome, s)].argmax(axis=1) for s in SEMENTES]
    lg, la = p_log.argmax(axis=1), P_LOG.argmax(axis=1)
    pares = {f"{nome} − rede base": list(zip(a, [PO[(BASE_R, s)].argmax(axis=1) for s in SEMENTES])),
             f"Dif({nome}): {nome} − logística com a mesma entrada": [(q, lg) for q in a],
             f"{nome} − logística atual": [(q, la) for q in a],
             f"logística com a entrada do {nome} − logística atual": [(lg, la)]}
    return tab, pd.DataFrame({n: {m: dif_media(p, m) for m in MET} for n, p in pares.items()}).T


tab_3, comp_3 = tabela_valor("R3", P_LOG_R3)
display(tab_3.round(3))
print("Diferença (média das 5 sementes) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):")
comp_3

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble,CIN → GS,GS → CIN
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757,17.6,8.2
R3,0.690,0.020,0.654,0.704,0.913,0.972,0.690,0.930,0.820,0.759,0.716,24.0,7.2
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN,18.0,6.0
"logística, entrada do R3",0.708,NaN,NaN,NaN,0.923,0.919,0.708,0.967,0.821,0.776,NaN,14.0,10.0


Diferença (média das 5 sementes) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):


,pior F1,F1 GS,F1 EBV,MCC
R3 − rede base,-0.028 (-0.062 a +0.008),-0.028 (-0.062 a +0.008),+0.007 (-0.023 a +0.041),-0.014 (-0.035 a +0.008)
Dif(R3): R3 − logística com a mesma entrada,-0.019 (-0.092 a +0.060),-0.019 (-0.092 a +0.060),-0.037 (-0.071 a -0.007),-0.002 (-0.046 a +0.044)
R3 − logística atual,-0.060 (-0.109 a -0.007),-0.060 (-0.109 a -0.007),-0.021 (-0.052 a +0.006),-0.023 (-0.056 a +0.014)
logística com a entrada do R3 − logística atual,-0.042 (-0.116 a +0.024),-0.042 (-0.116 a +0.024),+0.016 (+0.000 a +0.048),-0.021 (-0.061 a +0.017)


# R4 — compressão das caudas, 3·tanh(z/3), na mutação e na CNV (2026-11-04)

PARTE C de `docs/relatorio_imagem.txt` (R4); pré-registro no diário (2026-11-04, "Pré-registro do R4"). Rótulo DEFINIDO, o 3º da
fila (`.claude/rules/regras_de_testes.md`).

- **Hipótese:** sem o domínio dos amplicons e da variante isolada, a rede passa a usar os ganhos e as perdas largos, que separam
  o CIN do GS na regra do TCGA.
- **Entrada:** z por canal como hoje e, depois dele, z' = 3·tanh(z/3) na mutação e na CNV. A metilação fica como hoje.
- A logística com a mesma entrada roda antes da rede (`rodar_valor()`). O risco a acompanhar é o F1 do CIN.

In [29]:
def comprimir(z):
    # compressão das caudas (PARTE C, R4): perto de 0 quase não muda, e o módulo nunca passa de 3
    return 3 * np.tanh(z / 3)


def preparar_r4(k):
    # z por canal como hoje; depois, mutação e CNV comprimidas; a metilação fica como hoje
    z = padronizar(tensor, padronizacao[k])
    z[:, :2] = comprimir(z[:, :2])
    return z


# autoteste, antes da hora de treino. Contas à mão: 3·tanh(9,26/3) = 2,99; 3·tanh(7,6/3) = 2,96; 3·tanh(1/3) = 0,96
assert np.allclose(comprimir(np.array([9.26, 7.6, 1.0, 0.0, -9.26])), [2.99, 2.96, 0.96, 0.0, -2.99], atol=0.01)
for k in range(K):
    atual, z = padronizar(tensor, padronizacao[k]), preparar_r4(k)
    assert np.array_equal(z[:, 2], atual[:, 2]), k                    # metilação igual à de hoje
    assert np.array_equal(z[:, :2] == 0, atual[:, :2] == 0), k        # NaN → 0 continua 0
    assert (np.abs(z[:, :2]) <= 3).all(), k                           # nenhum valor passa de 3
    assert (np.abs(z[:, :2]) <= np.abs(atual[:, :2]) + 1e-6).all(), k  # só encolhe (folga do arredondamento em float32)
print("desvio por canal no treino do fold 4:", {c: round(float(z[fold != k, i].std()), 3) for i, c in enumerate(CANAIS)})
x = FORMATOS["hilbert"](z[sub])                                       # 1 época em 40 pacientes
p = treinar_variante(x, y[sub], x[:3], SEED, **BASE, epocas=1)
assert p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4)
del atual, z, x, p

desvio por canal no treino do fold 4: {'mutacao': 0.378, 'cnv': 0.802, 'metilacao': 0.976}


22:50:15 [RAM 1680 MB, pico 2010 MB]   1 épocas, melhor F1 macro na validação interna 0.200


In [30]:
P_LOG_R4 = rodar_valor("R4", preparar_r4)

22:50:16 [RAM 1544 MB, pico 2010 MB] === R4 ===
22:50:16 [RAM 1544 MB, pico 2010 MB] R4: semente 42


22:50:16 [RAM 1544 MB, pico 2010 MB] R4: semente 142


22:50:17 [RAM 1544 MB, pico 2010 MB] R4: semente 242


22:50:17 [RAM 1544 MB, pico 2010 MB] R4: semente 342


22:50:17 [RAM 1544 MB, pico 2010 MB] R4: semente 442


22:50:17 [RAM 1544 MB, pico 2010 MB] R4 terminado em 0 min


In [31]:
tab_4, comp_4 = tabela_valor("R4", P_LOG_R4)
display(tab_4.round(3))
print("Diferença (média das 5 sementes) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):")
comp_4

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble,CIN → GS,GS → CIN
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757,17.6,8.2
R4,0.706,0.031,0.667,0.750,0.922,0.972,0.706,0.941,0.835,0.800,0.742,16.8,10.0
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN,18.0,6.0
"logística, entrada do R4",0.686,NaN,NaN,NaN,0.910,0.901,0.686,0.951,0.795,0.775,NaN,18.0,9.0


Diferença (média das 5 sementes) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):


,pior F1,F1 GS,F1 EBV,MCC
R4 − rede base,-0.011 (-0.047 a +0.019),-0.011 (-0.047 a +0.019),+0.018 (-0.013 a +0.059),+0.001 (-0.018 a +0.020)
Dif(R4): R4 − logística com a mesma entrada,+0.020 (-0.044 a +0.087),+0.020 (-0.044 a +0.087),-0.010 (-0.037 a +0.018),+0.040 (-0.001 a +0.086)
R4 − logística atual,-0.044 (-0.102 a +0.007),-0.044 (-0.102 a +0.007),-0.010 (-0.037 a +0.018),-0.007 (-0.039 a +0.025)
logística com a entrada do R4 − logística atual,-0.064 (-0.126 a -0.012),-0.064 (-0.126 a -0.012),+0.000 (+0.000 a +0.000),-0.047 (-0.082 a -0.018)


# R5 — ladrilhos por cromossomo alinhados ao pooling (2026-11-05)

PARTE C de `docs/relatorio_imagem.txt` (R5); pré-registro no diário (2026-11-05, "Pré-registro do R5 e do R5-p"). Teste de arranjo: os valores
são os de hoje e só muda o pixel de cada gene. Um ladrilho por cromossomo (1–22 e X; por cromossomo, porque o P3b refutou o braço), de lado igual ao
menor múltiplo de 8 ≥ √genes, com a Hilbert local e em fileiras, num quadro de 96 × 288 (decisão do usuário, 2026-11-05). O R5-p usa o mesmo quadro,
com os genes sorteados entre as posições ocupadas pelo `PERM` do D2. A logística não muda, porque a entrada é a mesma.

In [32]:
def _gilbert(x, y, ax, ay, bx, by):
    # cópia do 03_images: curva de Hilbert generalizada para retângulos quaisquer (algoritmo "gilbert", J. Červený)
    w, h = abs(ax + ay), abs(bx + by)
    dax, day, dbx, dby = np.sign(ax), np.sign(ay), np.sign(bx), np.sign(by)
    if h == 1:
        for _ in range(w):
            yield x, y
            x, y = x + dax, y + day
        return
    if w == 1:
        for _ in range(h):
            yield x, y
            x, y = x + dbx, y + dby
        return
    ax2, ay2, bx2, by2 = ax // 2, ay // 2, bx // 2, by // 2
    w2, h2 = abs(ax2 + ay2), abs(bx2 + by2)
    if 2 * w > 3 * h:
        if w2 % 2 and w > 2:
            ax2, ay2 = ax2 + dax, ay2 + day
        yield from _gilbert(x, y, ax2, ay2, bx, by)
        yield from _gilbert(x + ax2, y + ay2, ax - ax2, ay - ay2, bx, by)
    else:
        if h2 % 2 and h > 2:
            bx2, by2 = bx2 + dbx, by2 + dby
        yield from _gilbert(x, y, bx2, by2, ax2, ay2)
        yield from _gilbert(x + bx2, y + by2, ax, ay, bx - bx2, by - by2)
        yield from _gilbert(x + (ax - dax) + (bx2 - dbx), y + (ay - day) + (by2 - dby),
                            -bx2, -by2, -(ax - ax2), -(ay - ay2))


def curva_hilbert(n):
    # cópia do 03_images
    lado = int(np.ceil(np.sqrt(n)))
    altura = int(np.ceil(n / lado))
    caminho = np.array(list(_gilbert(0, 0, lado, 0, 0, altura)))  # (x, y), começa no canto
    return caminho[:n, ::-1], (altura, lado)


def ladrilhos(largura):
    # tabela gene → pixel do R5: ladrilhos em ordem de cromossomo, em fileiras; lado = menor múltiplo de 8 ≥ √genes, porque os 3 MaxPool antes da
    # última convolução reduzem por 8; altura final arredondada ao múltiplo de 16, para os 4 MaxPool não descartarem linha
    pos, quadros, topo, esq, altura = np.empty((len(CROM), 2), int), {}, 0, 0, 0
    for c in CROMS:
        g = np.flatnonzero(CROM == c)  # os genes já estão em ordem genômica
        lado = 8 * int(np.ceil(np.sqrt(len(g)) / 8))
        if esq + lado > largura:
            topo, esq, altura = topo + altura, 0, 0
        pos[g] = curva_hilbert(len(g))[0] + [topo, esq]
        quadros[c] = (topo, esq, lado)
        esq, altura = esq + lado, max(altura, lado)
    return pos, quadros, (16 * int(np.ceil((topo + altura) / 16)), largura)


POS_R5, LADRILHOS, FORMA_R5 = ladrilhos(96)  # um layout só para os 5 folds: depende só da anotação


def para_ladrilhos(x):
    img = np.zeros((x.shape[0], x.shape[1], *FORMA_R5), np.float32)  # pixel vazio vale 0
    img[:, :, POS_R5[:, 0], POS_R5[:, 1]] = x
    return img


FORMATOS["ladrilhos"] = lambda z: np.moveaxis(para_ladrilhos(z), 1, -1)  # (n, 288, 96, 3)

# autoteste, antes das horas de treino. Tamanhos contados na PARTE C: cromossomo 1 = 48, 8 = 32, 18 = 16; soma das áreas 23.360
assert FORMA_R5 == (288, 96) and all(l % 16 == 0 for l in FORMA_R5) and len({q[0] for q in LADRILHOS.values()}) == 8
assert [LADRILHOS[c][2] for c in ("1", "8", "18")] == [48, 32, 16] and sum(q[2] ** 2 for q in LADRILHOS.values()) == 23360
grade = np.zeros(FORMA_R5, int)
for c, (t, e, l) in LADRILHOS.items():
    assert t % 8 == 0 and e % 8 == 0 and l % 8 == 0, c              # alinhado à grade de 8 pixels
    grade[t:t + l, e:e + l] += 1
    q = POS_R5[CROM == c]
    assert ((q >= [t, e]) & (q < [t + l, e + l])).all(), c          # os genes ficam no ladrilho do seu cromossomo
    passo = np.abs(np.diff(q, axis=0))                              # genes vizinhos no genoma: pixels vizinhos, com no máximo 1 passo em diagonal
    assert (passo.max(axis=1) == 1).all() and (passo.sum(axis=1) == 2).sum() <= 1, c  # (a curva dá esse passo com altura par e largura ímpar)
assert grade.max() == 1 and len(np.unique(POS_R5, axis=0)) == len(CROM)  # sem sobreposição; cada gene num pixel próprio
ocupado = np.zeros(FORMA_R5, bool)
ocupado[POS_R5[:, 0], POS_R5[:, 1]] = True
z0, g = padronizar(tensor[sub], padronizacao[0]), layout.gene.tolist().index("MLH1")
img, img_p = para_ladrilhos(z0), para_ladrilhos(z0[:, :, PERM[SEED]])
lin_g, col_g = POS_R5[g]
assert np.array_equal(img[:, :, lin_g, col_g], z0[:, :, g])                  # o pixel do MLH1 guarda o MLH1
assert np.array_equal(img_p[:, :, lin_g, col_g], z0[:, :, PERM[SEED][g]])    # R5-p: o mesmo pixel guarda outro gene
assert (img[:, :, ~ocupado] == 0).all() and (img_p[:, :, ~ocupado] == 0).all()  # R5-p: as mesmas posições ocupadas e vazias
assert rede((*FORMA_R5, 3)).count_params() == 61476                          # a média global mantém os parâmetros da rede base
x = FORMATOS["ladrilhos"](z0)                                                 # 1 época em 40 pacientes
p = treinar_variante(x, y[sub], x[:3], SEED, **BASE, epocas=1)
assert p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4)
print(f"quadro {FORMA_R5}: {len(CROM)} genes em {ocupado.size} pixels, {1 - ocupado.mean():.0%} vazios; lados: "
      f"{ {c: q[2] for c, q in LADRILHOS.items()} }")
del grade, q, passo, ocupado, z0, img, img_p, x, p

22:50:47 [RAM 1624 MB, pico 2010 MB]   1 épocas, melhor F1 macro na validação interna 0.071


quadro (288, 96): 17818 genes em 27648 pixels, 36% vazios; lados: {'1': 48, '2': 40, '3': 32, '4': 32, '5': 32, '6': 32, '7': 32, '8': 32, '9': 32, '10': 32, '11': 40, '12': 32, '13': 24, '14': 24, '15': 24, '16': 32, '17': 40, '18': 16, '19': 40, '20': 24, '21': 16, '22': 24, 'X': 32}


In [33]:
log("=== R5 e R5-p ===")
t0 = time.time()
for nome, sorteia in {"R5": False, "R5-p": True}.items():
    for s in SEMENTES:
        log(f"{nome}: semente {s}")
        PO[(nome, s)] = rodar_o(nome, s, BASE, "ladrilhos", preparar=permutado(s) if sorteia else None, pasta=OOF_R)
log(f"R5 e R5-p terminados em {(time.time() - t0) / 60:.0f} min")

22:50:48 [RAM 1626 MB, pico 2010 MB] === R5 e R5-p ===
22:50:48 [RAM 1626 MB, pico 2010 MB] R5: semente 42


22:50:48 [RAM 1626 MB, pico 2010 MB] R5: semente 142


22:50:49 [RAM 1626 MB, pico 2010 MB] R5: semente 242


22:50:49 [RAM 1626 MB, pico 2010 MB] R5: semente 342


22:50:49 [RAM 1626 MB, pico 2010 MB] R5: semente 442


22:50:49 [RAM 1626 MB, pico 2010 MB] R5-p: semente 42


22:50:49 [RAM 1626 MB, pico 2010 MB] R5-p: semente 142


22:50:50 [RAM 1626 MB, pico 2010 MB] R5-p: semente 242


22:50:50 [RAM 1626 MB, pico 2010 MB] R5-p: semente 342


22:50:50 [RAM 1626 MB, pico 2010 MB] R5-p: semente 442


22:50:50 [RAM 1626 MB, pico 2010 MB] R5 e R5-p terminados em 0 min


In [34]:
V5 = [BASE_R, "R5", "R5-p", "R0-p"]
tab_5 = resumo(V5)
tab_5["CIN → GS"] = [np.mean([cin_gs(PO[(v, s)]) for s in SEMENTES]) for v in V5] + [cin_gs(P_LOG)]
tab_5["GS → CIN"] = [np.mean([gs_cin(PO[(v, s)]) for s in SEMENTES]) for v in V5] + [gs_cin(P_LOG)]
tab_5.to_csv(RESULTS / "04_models_r5.csv")
display(tab_5.round(3))

comp5 = {"Perm(R5) = R5 − R5-p": ("R5", "R5-p"), "R5 − rede base": ("R5", BASE_R), "R5-p − R0-p": ("R5-p", "R0-p")}
print("Diferença pareada por semente (média das 5) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):")
pd.DataFrame({n: {m: dif_media(pares(a, b), m) for m in MET} for n, (a, b) in comp5.items()}).T

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble,CIN → GS,GS → CIN
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757,17.6,8.2
R5,0.713,0.026,0.688,0.742,0.921,0.958,0.713,0.894,0.825,0.742,0.776,17.4,9.2
R5-p,0.694,0.025,0.660,0.722,0.915,0.965,0.694,0.914,0.817,0.747,0.750,19.2,9.4
R0-p,0.697,0.037,0.642,0.731,0.908,0.949,0.697,0.882,0.802,0.765,0.729,22.8,7.0
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN,18.0,6.0


Diferença pareada por semente (média das 5) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):


,pior F1,F1 GS,F1 EBV,MCC
Perm(R5) = R5 − R5-p,+0.018 (-0.025 a +0.060),+0.018 (-0.025 a +0.060),-0.020 (-0.055 a +0.014),+0.007 (-0.017 a +0.032)
R5 − rede base,-0.005 (-0.036 a +0.024),-0.005 (-0.036 a +0.024),-0.029 (-0.068 a +0.009),-0.009 (-0.029 a +0.009)
R5-p − R0-p,-0.003 (-0.029 a +0.021),-0.003 (-0.029 a +0.021),+0.032 (-0.004 a +0.071),+0.015 (-0.000 a +0.033)


# R6 — genes ordenados por semelhança no treino do fold (2026-11-06)

PARTE C de `docs/relatorio_imagem.txt` (R6), com o redesenho aprovado pelo usuário; pré-registro no diário (2026-11-06, "Pré-registro do R6"). Teste de arranjo: os valores e o quadro são os
de hoje, e só muda o gene de cada pixel. O grafo de vizinhos da PARTE C não era conexo em nenhum fold (os grupos separados só têm genes do X, que não têm metilação), então os 825 genes do X saem do
embedding e vão para o fim da curva, em ordem genômica. Cada fold tem a sua ordem, ajustada só com o treino dele. O controle é o R0-p, já treinado. A logística não muda, porque a entrada é a mesma.

In [35]:
from scipy.sparse.csgraph import connected_components
from sklearn.manifold import SpectralEmbedding

AUT, EX = np.flatnonzero(CROM != "X"), np.flatnonzero(CROM == "X")  # os genes do X não têm metilação


def ordem_r6(k):
    # ORDEM[k]: pixel i → gene. Só o treino do fold k; cada autossomo é um ponto, CNV + metilação em z por gene (NaN → 0)
    z = padronizar_gene(tensor[fold != k], k, (1, 2))[:, 1:][:, :, AUT]  # (treino, 2, 16.993)
    emb = SpectralEmbedding(n_components=1, affinity="nearest_neighbors", n_neighbors=15, random_state=42)
    c = emb.fit_transform(np.concatenate([z[:, 0].T, z[:, 1].T], axis=1)).ravel()
    comp = connected_components(emb.affinity_matrix_, directed=False)[0]
    assert comp == 1, f"fold {k}: grafo com {comp} componentes"  # PARTE C: se não for conexo, a variante é redesenhada antes de treinar
    return np.r_[AUT[np.argsort(c, kind="stable")], EX]


ORDEM = {k: ordem_r6(k) for k in range(K)}
pd.DataFrame({f"fold{k}": layout.gene.to_numpy()[o] for k, o in ORDEM.items()}).to_csv(RESULTS / "r6_ordem.csv", index_label="pixel")

# autoteste, antes das horas de treino
assert len(AUT) == 16993 and len(EX) == 825
z0, g = padronizar(tensor[sub], padronizacao[0]), layout.gene.tolist().index("MLH1")
for k, o in ORDEM.items():
    assert np.array_equal(np.sort(o), np.arange(N)) and np.array_equal(o[-len(EX):], EX), k  # cada gene uma vez; o X no fim, em ordem genômica
i = np.flatnonzero(ORDEM[0] == g)[0]
assert i != g and np.array_equal(para_hilbert(z0[:, :, ORDEM[0]])[:, :, layout.linha.iloc[i], layout.coluna.iloc[i]], z0[:, :, g])  # o pixel i guarda o MLH1
assert rede((*FORMA, 3)).count_params() == 61476
x = FORMATOS["hilbert"](z0[:, :, ORDEM[0]])  # 1 época em 40 pacientes
p = treinar_variante(x, y[sub], x[:3], SEED, **BASE, epocas=1)
assert p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4)
print("pares vizinhos no mesmo cromossomo, por fold:", {k: round(float((CROM[o[1:]] == CROM[o[:-1]]).mean()), 3) for k, o in ORDEM.items()})
del z0, x, p

22:53:38 [RAM 1676 MB, pico 2010 MB]   1 épocas, melhor F1 macro na validação interna 0.071


pares vizinhos no mesmo cromossomo, por fold: {0: 0.857, 1: 0.874, 2: 0.912, 3: 0.902, 4: 0.891}


In [36]:
log("=== R6 ===")
t0 = time.time()
for s in SEMENTES:
    log(f"R6: semente {s}")
    PO[("R6", s)] = rodar_o("R6", s, BASE, "hilbert", preparar=lambda k: padronizar(tensor, padronizacao[k])[:, :, ORDEM[k]], pasta=OOF_R)
log(f"R6 terminado em {(time.time() - t0) / 60:.0f} min")

22:53:39 [RAM 1677 MB, pico 2010 MB] === R6 ===
22:53:39 [RAM 1677 MB, pico 2010 MB] R6: semente 42
22:53:39 [RAM 1677 MB, pico 2010 MB] R6 · semente 42 · fold 0


22:58:42 [RAM 2126 MB, pico 2449 MB]   24 épocas, melhor F1 macro na validação interna 0.825


22:58:45 [RAM 2003 MB, pico 2449 MB] R6 · semente 42 · fold 1


23:06:19 [RAM 2191 MB, pico 2568 MB]   37 épocas, melhor F1 macro na validação interna 0.967


23:06:22 [RAM 2083 MB, pico 2568 MB] R6 · semente 42 · fold 2


23:11:16 [RAM 2241 MB, pico 2581 MB]   24 épocas, melhor F1 macro na validação interna 0.917


... (41 outputs left out)

01:54:21 [RAM 2728 MB, pico 3034 MB]   16 épocas, melhor F1 macro na validação interna 0.890


01:54:22 [RAM 2605 MB, pico 3034 MB] R6 · semente 442 · fold 4


02:02:31 [RAM 2771 MB, pico 3048 MB]   36 épocas, melhor F1 macro na validação interna 0.855


02:02:38 [RAM 2648 MB, pico 3048 MB] R6 terminado em 189 min


In [37]:
V6 = [BASE_R, "R6", "R0-p"]
tab_6 = resumo(V6)
tab_6["CIN → GS"] = [np.mean([cin_gs(PO[(v, s)]) for s in SEMENTES]) for v in V6] + [cin_gs(P_LOG)]
tab_6["GS → CIN"] = [np.mean([gs_cin(PO[(v, s)]) for s in SEMENTES]) for v in V6] + [gs_cin(P_LOG)]
tab_6.to_csv(RESULTS / "04_models_r6.csv")
display(tab_6.round(3))

comp6 = {"Perm(R6) = R6 − R0-p": ("R6", "R0-p"), "R6 − rede base": ("R6", BASE_R)}
print("Diferença pareada por semente (média das 5) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):")
pd.DataFrame({n: {m: dif_media(pares(a, b), m) for m in MET} for n, (a, b) in comp6.items()}).T

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble,CIN → GS,GS → CIN
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757,17.6,8.2
R6,0.682,0.026,0.642,0.711,0.915,0.951,0.682,0.884,0.808,0.747,0.747,17.4,10.8
R0-p,0.697,0.037,0.642,0.731,0.908,0.949,0.697,0.882,0.802,0.765,0.729,22.8,7.0
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN,18.0,6.0


Diferença pareada por semente (média das 5) e IC 95% pelo bootstrap pareado (os 2.000 sorteios do 05_eval):


,pior F1,F1 GS,F1 EBV,MCC
Perm(R6) = R6 − R0-p,-0.016 (-0.056 a +0.022),-0.016 (-0.055 a +0.023),+0.002 (-0.025 a +0.035),+0.006 (-0.015 a +0.029)
R6 − rede base,-0.036 (-0.069 a -0.006),-0.036 (-0.068 a -0.006),-0.038 (-0.064 a -0.012),-0.026 (-0.041 a -0.010)
